<a href="https://colab.research.google.com/github/patienceobiemmanuel-cell/BEED-seizure-detection/blob/main/Copy_of_CS7P01_MSC_PROJECT_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CS7P01 MSc Project — EEG Seizure Detection Pipeline

In [ ]:
# ── 1. IMPORTS & GLOBAL CONFIGURATION ────────────────────────────────────────
import os, random, warnings, time, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as _cm
import matplotlib.colors as _mc
import seaborn as sns
import json, pickle, zipfile
from pathlib import Path
from scipy import stats
from scipy.fft import rfft, rfftfreq
from scipy.stats import wilcoxon, friedmanchisquare, spearmanr, ttest_rel
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, precision_score, recall_score,
    roc_auc_score, roc_curve, classification_report, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")

In [ ]:
# ── 2. PYTORCH ────────────────────────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    TORCH_AVAILABLE = True
    print(f"[INFO] PyTorch {torch.__version__} detected.")
except ImportError:
    TORCH_AVAILABLE = False
    print("[WARN] PyTorch not found. Install with: pip install torch")

# Global constants
SEED          = 42
N_FOLDS       = 5
SEGS_PER_SUB  = 200
N_CHANNELS    = 16
FS            = 128.0
WIN_DURATION  = 20.0
CLASS_NAMES   = ["Normal (C0)", "Pre-ictal (C1)", "Ictal (C2)", "Artefact (C3)"]
COLORS        = ["#2196F3", "#FF9800", "#F44336", "#9C27B0"]
FAR_THRESHOLD = 0.5
BATCH_SIZE    = 64
MAX_EPOCHS    = 40
PATIENCE      = 8
LR            = 1e-3
WD            = 1e-4
FOCAL_GAMMA   = 2.0
CLASS_WEIGHTS = [1.0, 1.2, 2.5, 3.5]
SAMPLE_W      = [1.0, 1.0, 2.0, 1.5]
ADV_WEIGHT    = 0.05

[INFO] PyTorch 2.10.0+cu128 detected.


In [ ]:
# ── 3. DATA PATH ──────────────────────────────────────────────────────────────
import os as _os

def _detect_data_path():
    candidates = [
        "/content/BEED_Data.csv",
        "/content/drive/MyDrive/BEED_Data.csv",
        "BEED_Data.csv",
        "../BEED_Data.csv",
    ]
    for p in candidates:
        if _os.path.exists(p):
            return p
    try:
        from google.colab import files as _cf
        print("[INFO] BEED_Data.csv not found. Upload it in the dialog below...")
        uploaded = _cf.upload()
        return list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError(
            "Cannot find BEED_Data.csv. Set DATA_PATH manually."
        )

DATA_PATH = _detect_data_path()
print(f"[INFO] Data file : {DATA_PATH}")

def _detect_out_dir():
    try:
        import google.colab
        p = Path("/content/outputs")
    except ImportError:
        p = Path("outputs")
    p.mkdir(parents=True, exist_ok=True)
    return p

OUT_DIR = _detect_out_dir()
print(f"[INFO] Output dir: {OUT_DIR}")

[INFO] BEED_Data.csv not found. Upload it in the dialog below...


Saving BEED_Data.csv to BEED_Data.csv
[INFO] Data file : BEED_Data.csv
[INFO] Output dir: /content/outputs


In [ ]:
# ── 4. SEED MANAGEMENT ────────────────────────────────────────────────────────
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()

In [ ]:
# ── 5. DATA LOADING & QUALITY VALIDATION ─────────────────────────────────────
def load_and_validate(path: str):
    df = pd.read_csv(path)
    channels = [f"X{i}" for i in range(1, N_CHANNELS + 1)]
    assert all(c in df.columns for c in channels), "Missing channel columns"
    assert "y" in df.columns,                      "Missing label column y"
    assert df["y"].nunique() == 4,                  "Expected 4 classes"
    assert df.isnull().sum().sum() == 0,            "Missing values detected"
    assert (df[channels].abs() <= 500).all().all(), "Outliers > ±500 µV"
    for ch in channels:
        assert df[ch].std() >= 0.01, f"Flat channel: {ch}"
    assert df.duplicated().sum() == 0,              "Duplicate rows found"
    vc = df["y"].value_counts()
    assert all(vc == 2000),                         "Unequal class sizes"
    df["subject_id"] = df.index // SEGS_PER_SUB
    n_subjects = df["subject_id"].nunique()
    assert n_subjects == 40, f"Expected 40 subjects, got {n_subjects}"
    subject_class = df.groupby("subject_id")["y"].nunique()
    assert (subject_class == 1).all(), "Each subject should belong to 1 class"
    print(f"[DATA] {df.shape[0]} segments × {N_CHANNELS} channels  |  40 subjects  |  4 balanced classes ✓")
    return df, channels

df, CHANNELS = load_and_validate(DATA_PATH)
X_raw  = df[CHANNELS].values.astype(np.float32)
y_all  = df["y"].values
sub_id = df["subject_id"].values

[DATA] 8000 segments × 16 channels  |  40 subjects  |  4 balanced classes ✓


In [ ]:
# ── 6. HELPER UTILITIES ───────────────────────────────────────────────────────
def save_fig(name: str, dpi: int = 150):
    path = OUT_DIR / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {path}")
    return path

def compute_far(y_true, y_pred, n_segs: int, seg_dur_s: float = WIN_DURATION):
    monitoring_hours = (n_segs * seg_dur_s) / 3600.0
    fp_count = np.sum((y_pred == 2) & (y_true != 2))
    return fp_count / monitoring_hours if monitoring_hours > 0 else np.nan

In [ ]:
# ── 7. EXPLORATORY DATA ANALYSIS ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("  EDA – EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# 7.1 Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("EDA – Class Distribution", fontsize=14, fontweight="bold")
counts = np.bincount(y_all)
bars = axes[0].bar(CLASS_NAMES, counts, color=COLORS, edgecolor="k", linewidth=0.7)
axes[0].set_title("Segment Count per Class")
axes[0].set_ylabel("Count")
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(cnt), ha="center", va="bottom", fontsize=9)
axes[1].pie(counts, labels=CLASS_NAMES, colors=COLORS, autopct="%1.1f%%",
            startangle=90, wedgeprops=dict(edgecolor="k", linewidth=0.7))
axes[1].set_title("Class Proportion")
plt.tight_layout()
save_fig("EDA_01_class_distribution")

# 7.2 Channel amplitude statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("EDA – Per-Class Channel Amplitude Statistics", fontsize=13, fontweight="bold")
for cls, ax in enumerate(axes.flatten()):
    Xc = X_raw[y_all == cls]
    means = Xc.mean(axis=0)
    stds  = Xc.std(axis=0)
    ax.fill_between(range(N_CHANNELS), means - stds, means + stds,
                    alpha=0.25, color=COLORS[cls], label="±1 SD")
    ax.plot(means, color=COLORS[cls], lw=2, marker="o", markersize=4, label="Mean")
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_title(CLASS_NAMES[cls], fontsize=11)
    ax.set_xlabel("EEG Channel (X1–X16)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_xticks(range(N_CHANNELS))
    ax.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=45, fontsize=7)
    ax.legend(fontsize=8)
plt.tight_layout()
save_fig("EDA_02_channel_amplitude_per_class")

# 7.3 Amplitude distribution violin
fig, ax = plt.subplots(figsize=(14, 5))
data_for_violin = [X_raw[y_all == c].flatten() for c in range(4)]
parts = ax.violinplot(data_for_violin, positions=range(4), showmedians=True, showextrema=True)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(COLORS[i])
    pc.set_alpha(0.6)
ax.set_xticks(range(4))
ax.set_xticklabels(CLASS_NAMES)
ax.set_title("EDA – Amplitude Distribution per Class (all channels pooled)", fontsize=12)
ax.set_ylabel("Amplitude (µV)")
ax.axhline(0, color="grey", lw=0.8, ls="--")
plt.tight_layout()
save_fig("EDA_03_amplitude_violin")

# 7.4 Inter-channel correlation heatmaps
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("EDA – Inter-channel Pearson Correlation per Class", fontsize=12, fontweight="bold")
for cls, ax in enumerate(axes):
    corr = np.corrcoef(X_raw[y_all == cls].T)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set_title(CLASS_NAMES[cls], fontsize=10)
    ax.set_xticks(range(N_CHANNELS))
    ax.set_yticks(range(N_CHANNELS))
    ax.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=90, fontsize=6)
    ax.set_yticklabels([f"X{i+1}" for i in range(N_CHANNELS)], fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
save_fig("EDA_04_correlation_heatmaps")

# 7.5 Subject-level amplitude spread
subject_means = np.array([X_raw[sub_id == s].mean() for s in range(40)])
subject_stds  = np.array([X_raw[sub_id == s].std()  for s in range(40)])
subject_class = np.array([y_all[sub_id == s][0] for s in range(40)])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EDA – Subject-level Amplitude Summary (40 subjects)", fontsize=12, fontweight="bold")
for cls in range(4):
    mask = subject_class == cls
    axes[0].scatter(np.where(mask)[0], subject_means[mask],
                    color=COLORS[cls], label=CLASS_NAMES[cls], s=50, zorder=3)
    axes[1].scatter(np.where(mask)[0], subject_stds[mask],
                    color=COLORS[cls], s=50, zorder=3)
for ax, lbl in zip(axes, ["Mean Amplitude (µV)", "Std Amplitude (µV)"]):
    ax.set_xlabel("Subject Index (0–39)")
    ax.set_ylabel(lbl)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_title("Mean per Subject")
axes[1].set_title("Std per Subject")
plt.tight_layout()
save_fig("EDA_05_subject_level_summary")

# 7.6 Box plots per channel
fig, axes = plt.subplots(4, 1, figsize=(16, 14))
fig.suptitle("EDA – Channel Amplitude Box Plots per Class", fontsize=13, fontweight="bold")
for cls, ax in enumerate(axes):
    Xc = X_raw[y_all == cls]
    ax.boxplot(Xc.T.tolist(), notch=True, patch_artist=True,
               boxprops=dict(facecolor=COLORS[cls], alpha=0.5),
               medianprops=dict(color="k", linewidth=2))
    ax.set_title(CLASS_NAMES[cls], fontsize=10)
    ax.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], fontsize=8)
    ax.set_ylabel("Amplitude (µV)")
    ax.axhline(0, color="grey", lw=0.8, ls="--")
plt.tight_layout()
save_fig("EDA_06_channel_boxplots")

# 7.7 Kurtosis & skewness
kurt_arr = np.array([[stats.kurtosis(X_raw[y_all == c, ch]) for ch in range(N_CHANNELS)]
                      for c in range(4)])
skew_arr = np.array([[stats.skew(X_raw[y_all == c, ch]) for ch in range(N_CHANNELS)]
                      for c in range(4)])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("EDA – Statistical Shape Features per Class", fontsize=12, fontweight="bold")
for ax, arr, lbl in zip(axes, [kurt_arr, skew_arr], ["Kurtosis", "Skewness"]):
    for cls in range(4):
        ax.plot(arr[cls], marker="o", lw=1.5, color=COLORS[cls],
                label=CLASS_NAMES[cls], markersize=4)
    ax.set_xticks(range(N_CHANNELS))
    ax.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=45, fontsize=7)
    ax.set_title(lbl)
    ax.set_ylabel(lbl + " value")
    ax.legend(fontsize=8)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.grid(alpha=0.3)
plt.tight_layout()
save_fig("EDA_07_kurtosis_skewness")
print("[EDA] All 7 EDA figures saved.\n")


  EDA – EXPLORATORY DATA ANALYSIS
[SAVED] /content/outputs/EDA_01_class_distribution.png
[SAVED] /content/outputs/EDA_02_channel_amplitude_per_class.png
[SAVED] /content/outputs/EDA_03_amplitude_violin.png
[SAVED] /content/outputs/EDA_04_correlation_heatmaps.png
[SAVED] /content/outputs/EDA_05_subject_level_summary.png
[SAVED] /content/outputs/EDA_06_channel_boxplots.png
[SAVED] /content/outputs/EDA_07_kurtosis_skewness.png
[EDA] All 7 EDA figures saved.



In [ ]:
# ── 8. FFT SPECTRAL vs RAW TEMPORAL REPRESENTATION ───────────────────────────
print("=" * 70)
print("  REPRESENTATION COMPARISON – FFT SPECTRAL vs RAW TEMPORAL")
print("=" * 70)

BANDS = {
    "Delta (0.5–4 Hz)":  (0.5,   4.0),
    "Theta (4–8 Hz)":    (4.0,   8.0),
    "Alpha (8–13 Hz)":   (8.0,  13.0),
    "Beta (13–30 Hz)":   (13.0, 30.0),
    "Gamma (30–128 Hz)": (30.0, 128.0),
}

def extract_fft_features(X: np.ndarray) -> np.ndarray:
    N, C = X.shape
    freqs = rfftfreq(C, d=1.0 / FS)
    feats = []
    for row in X:
        fft_pow = np.abs(rfft(row)) ** 2
        band_powers = []
        for lo, hi in BANDS.values():
            mask = (freqs >= lo) & (freqs < hi)
            bp   = fft_pow[mask].mean() if mask.any() else 0.0
            band_powers.append(bp)
        feats.append(band_powers)
    return np.array(feats, dtype=np.float32)

def extract_fft_features_full(X: np.ndarray) -> np.ndarray:
    N, C = X.shape
    freqs = rfftfreq(C, d=1.0 / FS)
    feats = []
    for row in X:
        row_feats = []
        for ch in range(C):
            single = np.zeros(C); single[ch] = row[ch]
            fft_pow = np.abs(rfft(row)) ** 2
            for lo, hi in BANDS.values():
                mask = (freqs >= lo) & (freqs < hi)
                row_feats.append(fft_pow[mask].mean() if mask.any() else 0.0)
            break
        feats.append(row_feats)
    return np.array(feats, dtype=np.float32)

X_fft = extract_fft_features(X_raw)
band_names = list(BANDS.keys())

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Representation Comparison – FFT Power Spectrum vs Raw Temporal Signal",
             fontsize=13, fontweight="bold")
freqs_full = rfftfreq(N_CHANNELS, d=1.0 / FS)
for cls in range(4):
    idx = np.where(y_all == cls)[0][0]
    seg = X_raw[idx]
    ax = axes[0, cls]
    fft_pow = np.abs(rfft(seg)) ** 2
    ax.stem(freqs_full, fft_pow, linefmt=COLORS[cls], markerfmt=".", basefmt="grey", label="Power")
    for bname, (lo, hi) in BANDS.items():
        ax.axvspan(lo, hi, alpha=0.08)
    ax.set_title(f"{CLASS_NAMES[cls]}\nFFT Power Spectrum", fontsize=9)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power (µV²)")
    ax.set_xlim(0, FS / 2)
    ax2 = axes[1, cls]
    ax2.plot(range(N_CHANNELS), seg, color=COLORS[cls], lw=2, marker="o", markersize=4)
    ax2.fill_between(range(N_CHANNELS), seg, alpha=0.15, color=COLORS[cls])
    ax2.axhline(0, color="grey", lw=0.8, ls="--")
    ax2.set_title(f"Raw Spatial Amplitude\n(16 channels)", fontsize=9)
    ax2.set_xlabel("Channel Index")
    ax2.set_ylabel("Amplitude (µV)")
    ax2.set_xticks(range(N_CHANNELS))
    ax2.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=90, fontsize=6)
plt.tight_layout()
save_fig("REP_01_fft_vs_raw_temporal_comparison")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Representation Comparison – Mean FFT Band Power per Class", fontsize=12)
for cls in range(4):
    mean_bp = X_fft[y_all == cls].mean(axis=0)
    axes[0].bar(np.arange(5) + cls * 0.2, mean_bp, width=0.18,
                label=CLASS_NAMES[cls], color=COLORS[cls], alpha=0.8, edgecolor="k", lw=0.5)
axes[0].set_xticks(np.arange(5) + 0.3)
axes[0].set_xticklabels(band_names, rotation=20, ha="right", fontsize=8)
axes[0].set_ylabel("Mean Band Power (µV²)")
axes[0].set_title("FFT Band Powers (5 features per segment)")
axes[0].legend(fontsize=8)
for cls in range(4):
    axes[1].plot(X_raw[y_all == cls].mean(axis=0), color=COLORS[cls],
                 lw=2, marker="o", markersize=4, label=CLASS_NAMES[cls])
axes[1].axhline(0, color="grey", lw=0.8, ls="--")
axes[1].set_xticks(range(N_CHANNELS))
axes[1].set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=45, fontsize=7)
axes[1].set_ylabel("Mean Amplitude (µV)")
axes[1].set_title("Raw Temporal Profile (16 spatial points)")
axes[1].legend(fontsize=8)
plt.tight_layout()
save_fig("REP_02_fft_bandpower_vs_raw_mean_profile")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Representation Comparison – Class 0 vs Class 3 Boundary\n"
             "(FFT flattens temporal propagation differences)", fontsize=12, fontweight="bold")
for ci, cls in enumerate([0, 3]):
    fft_means = X_fft[y_all == cls].mean(axis=0)
    fft_stds  = X_fft[y_all == cls].std(axis=0)
    ax = axes[0, ci]
    ax.bar(range(5), fft_means, yerr=fft_stds, capsize=4,
           color=COLORS[cls], alpha=0.7, edgecolor="k", lw=0.7)
    ax.set_xticks(range(5))
    ax.set_xticklabels(band_names, rotation=20, ha="right", fontsize=7)
    ax.set_title(f"FFT Band Power – {CLASS_NAMES[cls]}")
    ax.set_ylabel("Power (µV²)")
    raw_means = X_raw[y_all == cls].mean(axis=0)
    raw_stds  = X_raw[y_all == cls].std(axis=0)
    ax2 = axes[1, ci]
    ax2.fill_between(range(N_CHANNELS), raw_means - raw_stds, raw_means + raw_stds,
                     alpha=0.2, color=COLORS[cls])
    ax2.plot(raw_means, color=COLORS[cls], lw=2, marker="o", markersize=4)
    ax2.axhline(0, color="grey", lw=0.8, ls="--")
    ax2.set_xticks(range(N_CHANNELS))
    ax2.set_xticklabels([f"X{i+1}" for i in range(N_CHANNELS)], rotation=45, fontsize=7)
    ax2.set_title(f"Raw Temporal Profile – {CLASS_NAMES[cls]}")
    ax2.set_ylabel("Amplitude (µV)")
plt.tight_layout()
save_fig("REP_03_class0_vs_class3_boundary")
print("[REP] Representation comparison figures saved.\n")

  REPRESENTATION COMPARISON – FFT SPECTRAL vs RAW TEMPORAL
[SAVED] /content/outputs/REP_01_fft_vs_raw_temporal_comparison.png
[SAVED] /content/outputs/REP_02_fft_bandpower_vs_raw_mean_profile.png
[SAVED] /content/outputs/REP_03_class0_vs_class3_boundary.png
[REP] Representation comparison figures saved.



In [ ]:
# ── 9. CHANNEL-WISE Z-SCORE NORMALISATION ────────────────────────────────────
print("=" * 70)
print("  CHANNEL-WISE Z-SCORE NORMALISATION (Fold-level, leakage-free)")
print("=" * 70)

def channelwise_zscore(X_train: np.ndarray, X_val: np.ndarray, eps: float = 1e-8):
    mu    = X_train.mean(axis=0, keepdims=True)
    sigma = X_train.std(axis=0, keepdims=True) + eps
    return (X_train - mu) / sigma, (X_val - mu) / sigma, mu, sigma

gkf    = GroupKFold(n_splits=N_FOLDS)
splits = list(gkf.split(X_raw, y_all, groups=sub_id))
tr_idx, va_idx = splits[0]
X_tr_demo  = X_raw[tr_idx]
X_va_demo  = X_raw[va_idx]
X_tr_norm, X_va_norm, mu_demo, sig_demo = channelwise_zscore(X_tr_demo, X_va_demo)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Channel-wise Z-Score Normalisation (Fold 1 Training Split)\n"
             "Statistics computed from training data only — no leakage",
             fontsize=12, fontweight="bold")
axes[0, 0].hist(X_tr_demo.flatten(), bins=80, color="#607D8B", alpha=0.7, edgecolor="w", lw=0.3)
axes[0, 0].set_title("Before Norm – Training (all channels)")
axes[0, 0].set_xlabel("Amplitude (µV)"); axes[0, 0].set_ylabel("Count")
axes[0, 0].axvline(X_tr_demo.mean(), color="red", ls="--", lw=1.5, label=f"μ={X_tr_demo.mean():.2f}")
axes[0, 0].legend()
axes[0, 1].hist(X_tr_norm.flatten(), bins=80, color="#4CAF50", alpha=0.7, edgecolor="w", lw=0.3)
axes[0, 1].set_title("After Norm – Training")
axes[0, 1].set_xlabel("Normalised amplitude"); axes[0, 1].set_ylabel("Count")
axes[0, 1].axvline(X_tr_norm.mean(), color="red", ls="--", lw=1.5, label=f"μ={X_tr_norm.mean():.4f}")
axes[0, 1].legend()
axes[0, 2].hist(X_va_norm.flatten(), bins=80, color="#FF5722", alpha=0.7, edgecolor="w", lw=0.3)
axes[0, 2].set_title("After Norm – Validation\n(training stats applied)")
axes[0, 2].set_xlabel("Normalised amplitude"); axes[0, 2].set_ylabel("Count")
axes[0, 2].axvline(X_va_norm.mean(), color="k", ls="--", lw=1.5, label=f"μ={X_va_norm.mean():.4f}")
axes[0, 2].legend()
ch_labels = [f"X{i+1}" for i in range(N_CHANNELS)]
axes[1, 0].bar(range(N_CHANNELS), mu_demo.flatten(), color="#607D8B", alpha=0.8)
axes[1, 0].set_title("Per-channel Mean (from training)")
axes[1, 0].set_xticks(range(N_CHANNELS)); axes[1, 0].set_xticklabels(ch_labels, rotation=90, fontsize=7)
axes[1, 0].set_ylabel("Mean (µV)")
axes[1, 1].bar(range(N_CHANNELS), sig_demo.flatten(), color="#9C27B0", alpha=0.8)
axes[1, 1].set_title("Per-channel Std (from training)")
axes[1, 1].set_xticks(range(N_CHANNELS)); axes[1, 1].set_xticklabels(ch_labels, rotation=90, fontsize=7)
axes[1, 1].set_ylabel("Std (µV)")
axes[1, 2].bar(range(N_CHANNELS), X_tr_norm.std(axis=0), color="#4CAF50", alpha=0.8, label="Train std")
axes[1, 2].bar(range(N_CHANNELS), X_va_norm.std(axis=0), color="#FF5722", alpha=0.5, label="Val std")
axes[1, 2].set_title("Post-norm Std per Channel")
axes[1, 2].set_xticks(range(N_CHANNELS)); axes[1, 2].set_xticklabels(ch_labels, rotation=90, fontsize=7)
axes[1, 2].set_ylabel("Std")
axes[1, 2].legend(fontsize=8)
plt.tight_layout()
save_fig("NORM_01_zscore_normalisation")
print("[NORM] Z-score normalisation figure saved.\n")

  CHANNEL-WISE Z-SCORE NORMALISATION (Fold-level, leakage-free)
[SAVED] /content/outputs/NORM_01_zscore_normalisation.png
[NORM] Z-score normalisation figure saved.



In [ ]:
# ── 10. PREPROCESSING OUTPUT VISUALISATION ───────────────────────────────────
print("=" * 70)
print("  PREPROCESSING OUTPUT VISUALISATION")
print("=" * 70)

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
fig.suptitle("Preprocessing Output: Raw EEG Amplitude → Normalised (per class example)",
             fontsize=12, fontweight="bold")
for cls in range(4):
    local_idx = np.where(y_all[tr_idx] == cls)[0][0]
    raw_seg  = X_tr_demo[local_idx]
    norm_seg = X_tr_norm[local_idx]
    ax_raw  = axes[cls, 0]
    ax_norm = axes[cls, 1]
    ax_raw.plot(range(N_CHANNELS), raw_seg,  color=COLORS[cls], lw=2, marker="o", markersize=5)
    ax_raw.fill_between(range(N_CHANNELS), raw_seg, alpha=0.15, color=COLORS[cls])
    ax_raw.axhline(0, color="grey", lw=0.8, ls="--")
    ax_raw.set_title(f"{CLASS_NAMES[cls]} – Raw Signal", fontsize=10)
    ax_raw.set_ylabel("Amplitude (µV)")
    ax_raw.set_xticks(range(N_CHANNELS))
    ax_raw.set_xticklabels(ch_labels, rotation=90, fontsize=7)
    ax_norm.plot(range(N_CHANNELS), norm_seg, color=COLORS[cls], lw=2, marker="s", markersize=5)
    ax_norm.fill_between(range(N_CHANNELS), norm_seg, alpha=0.15, color=COLORS[cls])
    ax_norm.axhline(0, color="grey", lw=0.8, ls="--")
    ax_norm.axhline( 1, color="green", lw=0.8, ls=":", alpha=0.6)
    ax_norm.axhline(-1, color="green", lw=0.8, ls=":", alpha=0.6)
    ax_norm.set_title(f"{CLASS_NAMES[cls]} – Normalised (z-score)", fontsize=10)
    ax_norm.set_ylabel("z-score")
    ax_norm.set_xticks(range(N_CHANNELS))
    ax_norm.set_xticklabels(ch_labels, rotation=90, fontsize=7)
plt.tight_layout()
save_fig("PREPROC_01_raw_vs_normalised_per_class")

sample_idx = np.concatenate([np.where(y_all[tr_idx] == c)[0][:50] for c in range(4)])
X_sample   = X_tr_norm[sample_idx]
y_sample   = y_all[tr_idx][sample_idx]
sort_order  = np.argsort(y_sample)
X_sample   = X_sample[sort_order]
y_sample   = y_sample[sort_order]
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(X_sample.T, aspect="auto", cmap="seismic", vmin=-3, vmax=3, interpolation="nearest")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label="z-score")
boundaries = [0, 50, 100, 150, 200]
for b in boundaries:
    ax.axvline(b - 0.5, color="lime", lw=2)
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(ch_labels, fontsize=8)
ax.set_xlabel("Segment Index (50 per class, sorted by class)")
ax.set_ylabel("EEG Channel")
ax.set_title("Preprocessing Output: Normalised EEG Matrix (X_norm)\n"
             "Columns: segments  |  Rows: channels  |  Lime lines: class boundaries", fontsize=11)
for cls in range(4):
    ax.text(25 + cls * 50, -0.8, CLASS_NAMES[cls], ha="center", va="top",
            fontsize=8, fontweight="bold", color=COLORS[cls])
plt.tight_layout()
save_fig("PREPROC_02_normalised_eeg_heatmap")

fig, axes = plt.subplots(1, N_FOLDS, figsize=(16, 4), sharey=True)
fig.suptitle("Preprocessing Output: Class Distribution per Subject-Stratified Fold",
             fontsize=12, fontweight="bold")
for fi, (tr_i, va_i) in enumerate(splits):
    counts_tr = np.bincount(y_all[tr_i], minlength=4)
    counts_va = np.bincount(y_all[va_i], minlength=4)
    ax = axes[fi]
    x = np.arange(4)
    ax.bar(x - 0.2, counts_tr, 0.35, label="Train", color="#4CAF50", alpha=0.8, edgecolor="k")
    ax.bar(x + 0.2, counts_va, 0.35, label="Val",   color="#FF5722", alpha=0.8, edgecolor="k")
    ax.set_title(f"Fold {fi+1}")
    ax.set_xticks(x)
    ax.set_xticklabels(["C0","C1","C2","C3"], fontsize=9)
    if fi == 0:
        ax.set_ylabel("Segments")
    ax.legend(fontsize=7)
plt.tight_layout()
save_fig("PREPROC_03_fold_class_distribution")
print("[PREPROC] Preprocessing output figures saved.\n")

  PREPROCESSING OUTPUT VISUALISATION
[SAVED] /content/outputs/PREPROC_01_raw_vs_normalised_per_class.png
[SAVED] /content/outputs/PREPROC_02_normalised_eeg_heatmap.png
[SAVED] /content/outputs/PREPROC_03_fold_class_distribution.png
[PREPROC] Preprocessing output figures saved.



In [ ]:
# ── 11. SEIZURE ONSET ANNOTATION ─────────────────────────────────────────────
print("=" * 70)
print("  SEIZURE ONSET ANNOTATION – Single Channel")
print("=" * 70)

def synthesize_eeg_trace(amplitude_µV: float, cls: int, n_points: int = 512,
                          fs_sim: float = 256.0, rng: np.random.Generator = None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    t = np.linspace(0, WIN_DURATION, n_points)
    base_freq = {0: 10.0, 1: 4.0, 2: 3.0, 3: 12.0}[cls]
    amp_scale = {0: 0.3,  1: 0.6, 2: 1.2, 3: 0.8 }[cls]
    noise     = {0: 0.15, 1: 0.25, 2: 0.10, 3: 0.30}[cls]
    signal_v  = amp_scale * np.sin(2 * np.pi * base_freq * t)
    if cls == 1:
        ramp = np.linspace(0.3, 1.0, n_points)
        signal_v *= ramp
        signal_v += 0.3 * np.sin(2 * np.pi * 8 * t)
    elif cls == 2:
        burst_start = int(0.3 * n_points)
        envelope = np.ones(n_points) * 0.2
        envelope[burst_start:] = 1.0
        signal_v = envelope * (np.sin(2 * np.pi * 3 * t) + 0.5 * np.sin(2 * np.pi * 6 * t))
        signal_v[:burst_start] = rng.normal(0, 0.2, burst_start)
    elif cls == 3:
        spike_times = [0.15, 0.45, 0.72]
        for st in spike_times:
            idx = int(st * n_points)
            width = int(0.02 * n_points)
            w = np.exp(-0.5 * ((np.arange(n_points) - idx) / width) ** 2)
            signal_v += 1.5 * w * rng.choice([-1, 1])
    signal_v += rng.normal(0, noise, n_points)
    if signal_v.std() > 0:
        signal_v = signal_v / signal_v.std() * (abs(amplitude_µV) + 5)
    return t, signal_v

cls_target = 1
ch_target  = 7
seg_target = np.where(y_all == cls_target)[0][100]
rng_demo = np.random.default_rng(SEED)
amplitude = X_raw[seg_target, ch_target]
t_sim, sig_sim = synthesize_eeg_trace(amplitude, cls_target, rng=rng_demo)
seg_normal = np.where(y_all == 0)[0][50]
amp_normal = X_raw[seg_normal, ch_target]
_, sig_normal = synthesize_eeg_trace(amp_normal, 0, rng=rng_demo)

fig, ax = plt.subplots(figsize=(14, 5))
seizure_onset_t = WIN_DURATION * 0.30
ax.axvspan(0, seizure_onset_t, alpha=0.08, color="#2196F3", label="Background (pre-onset)")
ax.axvspan(seizure_onset_t, WIN_DURATION, alpha=0.08, color="#FF9800", label="Seizure onset region")
ax.plot(t_sim, sig_sim, color=COLORS[1], lw=1.3, label="EEG – Pre-ictal (Channel X8)")
onset_amp = sig_sim[int(0.30 * len(sig_sim))]
ax.annotate("▶  SEIZURE ONSET\n(Class 1 – Pre-ictal begins)",
    xy=(seizure_onset_t, onset_amp),
    xytext=(seizure_onset_t + 1.5, onset_amp + abs(onset_amp) * 0.8 + 20),
    fontsize=10, fontweight="bold", color="#E65100",
    arrowprops=dict(arrowstyle="->", color="#E65100", lw=2.0, connectionstyle="arc3,rad=-0.3"),
    bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="#E65100", lw=1.5))
ax.axvline(seizure_onset_t, color="#E65100", lw=2, ls="--", alpha=0.85)
peak_idx = np.argmax(np.abs(sig_sim[int(0.3 * len(sig_sim)):]))
peak_t   = t_sim[int(0.3 * len(sig_sim)) + peak_idx]
peak_a   = sig_sim[int(0.3 * len(sig_sim)) + peak_idx]
ax.annotate(f"Peak: {peak_a:.1f} µV",
    xy=(peak_t, peak_a), xytext=(peak_t - 2.5, peak_a * 1.25),
    fontsize=9, color="#B71C1C",
    arrowprops=dict(arrowstyle="->", color="#B71C1C"),
    bbox=dict(boxstyle="round", fc="mistyrose", ec="#B71C1C"))
ax.set_title("Seizure Onset Annotation – Channel X8 (Pre-ictal EEG)\n"
    "Segment ID: {seg_target}  |  Class 1 (Pre-ictal)  |  "
    "Recorded amplitude: {amp:.1f} µV".format(seg_target=seg_target, amp=amplitude),
    fontsize=11, fontweight="bold")
ax.set_xlabel("Time within 20-second EEG window (seconds)")
ax.set_ylabel("Amplitude (µV)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.25)
props = dict(boxstyle="round", facecolor="lightyellow", alpha=0.8)
info = (f"Segment: {seg_target}\nChannel: X8 (idx 7)\n"
        f"Class: {CLASS_NAMES[1]}\nAmplitude: {amplitude:.1f} µV\nOnset @ {seizure_onset_t:.1f} s")
ax.text(0.99, 0.97, info, transform=ax.transAxes, fontsize=8, va="top", ha="right", bbox=props)
plt.tight_layout()
save_fig("ANNOT_01_seizure_onset_single_channel")

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
fig.suptitle("EEG Signal Morphology – All Four Classes (Channel X8)\nSeizure onset annotated in Pre-ictal (Class 1)",
             fontsize=13, fontweight="bold")
rng2 = np.random.default_rng(SEED + 1)
for cls in range(4):
    idx_  = np.where(y_all == cls)[0][80]
    amp_  = X_raw[idx_, ch_target]
    t_, s_ = synthesize_eeg_trace(amp_, cls, rng=rng2)
    ax = axes[cls]
    ax.plot(t_, s_, color=COLORS[cls], lw=1.4, label=f"{CLASS_NAMES[cls]}  |  Amp: {amp_:.1f} µV")
    ax.fill_between(t_, s_, alpha=0.12, color=COLORS[cls])
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_ylabel("Amplitude (µV)", fontsize=9)
    ax.legend(loc="upper right", fontsize=9, framealpha=0.8)
    ax.grid(alpha=0.2)
    if cls == 1:
        ax.axvspan(WIN_DURATION * 0.30, WIN_DURATION, alpha=0.07, color="#FF9800")
        ax.axvline(WIN_DURATION * 0.30, color="#E65100", lw=2, ls="--")
        ax.annotate("SEIZURE ONSET →",
                    xy=(WIN_DURATION * 0.30, s_[int(0.30 * len(s_))]),
                    xytext=(WIN_DURATION * 0.05, s_.max() * 0.7),
                    fontsize=9, fontweight="bold", color="#E65100",
                    arrowprops=dict(arrowstyle="->", color="#E65100"),
                    bbox=dict(boxstyle="round", fc="lightyellow", ec="#E65100"))
    elif cls == 3:
        spike_idx = np.argmax(np.abs(s_))
        ax.annotate("Paroxysmal\nArtefact Spike",
                    xy=(t_[spike_idx], s_[spike_idx]),
                    xytext=(t_[spike_idx] + 2, s_[spike_idx] * 0.6),
                    fontsize=8, color="#6A1B9A",
                    arrowprops=dict(arrowstyle="->", color="#6A1B9A"),
                    bbox=dict(boxstyle="round", fc="#E1BEE7", ec="#6A1B9A"))
axes[-1].set_xlabel("Time within 20-second EEG window (seconds)")
plt.tight_layout()
save_fig("ANNOT_02_all_classes_with_onset")
print("[ANNOT] Seizure onset annotation figures saved.\n")

  SEIZURE ONSET ANNOTATION – Single Channel
[SAVED] /content/outputs/ANNOT_01_seizure_onset_single_channel.png
[SAVED] /content/outputs/ANNOT_02_all_classes_with_onset.png
[ANNOT] Seizure onset annotation figures saved.



In [ ]:
# ── 12. SUBJECT-STRATIFIED 5-FOLD CV SETUP ───────────────────────────────────
print("=" * 70)
print("  SUBJECT-STRATIFIED 5-FOLD CV + FEATURE PREPARATION")
print("=" * 70)

fold_data = []
for fold_idx, (tr_idx, va_idx) in enumerate(splits):
    X_tr, X_va, mu_f, sig_f = channelwise_zscore(X_raw[tr_idx], X_raw[va_idx])
    X_tr_fft = extract_fft_features(X_tr)
    X_va_fft = extract_fft_features(X_va)
    fold_data.append({
        "tr_idx": tr_idx, "va_idx": va_idx,
        "X_tr":   X_tr,   "X_va":   X_va,
        "y_tr":   y_all[tr_idx], "y_va": y_all[va_idx],
        "X_tr_fft": X_tr_fft,   "X_va_fft": X_va_fft,
        "mu": mu_f, "sigma": sig_f,
    })
    n_sub_tr = len(np.unique(sub_id[tr_idx]))
    n_sub_va = len(np.unique(sub_id[va_idx]))
    overlap  = set(sub_id[tr_idx]) & set(sub_id[va_idx])
    print(f"  Fold {fold_idx+1}: {len(tr_idx)} train segs / {len(va_idx)} val segs "
          f"| {n_sub_tr} / {n_sub_va} subjects | overlap = {len(overlap)} subjects ✓")

  SUBJECT-STRATIFIED 5-FOLD CV + FEATURE PREPARATION
  Fold 1: 6400 train segs / 1600 val segs | 32 / 8 subjects | overlap = 0 subjects ✓
  Fold 2: 6400 train segs / 1600 val segs | 32 / 8 subjects | overlap = 0 subjects ✓
  Fold 3: 6400 train segs / 1600 val segs | 32 / 8 subjects | overlap = 0 subjects ✓
  Fold 4: 6400 train segs / 1600 val segs | 32 / 8 subjects | overlap = 0 subjects ✓
  Fold 5: 6400 train segs / 1600 val segs | 32 / 8 subjects | overlap = 0 subjects ✓


In [ ]:
# ── 13. CLASSICAL ML MODELS ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  CLASSICAL ML MODELS: SVM · RF · XGBoost  (FFT features)")
print("=" * 70)

def evaluate_classical(y_true, y_pred, y_prob, n_segs, model_name, fold):
    acc  = accuracy_score(y_true, y_pred)
    mf1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(label_binarize(y_true, classes=[0,1,2,3]),
                             y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    far  = compute_far(y_true, y_pred, n_segs)
    return {"model": model_name, "fold": fold+1,
            "acc": acc, "macro_f1": mf1, "auroc": auc, "FAR_hr": far}

classical_results = []
for fold_idx, fd in enumerate(fold_data):
    X_tr_f = fd["X_tr_fft"]; X_va_f = fd["X_va_fft"]
    y_tr   = fd["y_tr"];      y_va   = fd["y_va"]
    n_va   = len(y_va)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_f)
    X_va_s = scaler.transform(X_va_f)

    set_seed()
    svm = SVC(C=10, kernel="rbf", gamma="scale", probability=True, random_state=SEED)
    svm.fit(X_tr_s, y_tr)
    svm_pred = svm.predict(X_va_s)
    svm_prob = svm.predict_proba(X_va_s)
    svm_res = evaluate_classical(y_va, svm_pred, svm_prob, n_va, "SVM", fold_idx)
    classical_results.append(svm_res)

    set_seed()
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_s, y_tr)
    rf_pred = rf.predict(X_va_s)
    rf_prob  = rf.predict_proba(X_va_s)
    rf_res = evaluate_classical(y_va, rf_pred, rf_prob, n_va, "RF", fold_idx)
    classical_results.append(rf_res)

    set_seed()
    xgb_clf = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                  eval_metric="mlogloss", random_state=SEED,
                                  use_label_encoder=False, verbosity=0)
    xgb_clf.fit(X_tr_s, y_tr)
    xgb_pred = xgb_clf.predict(X_va_s)
    xgb_prob  = xgb_clf.predict_proba(X_va_s)
    xgb_res = evaluate_classical(y_va, xgb_pred, xgb_prob, n_va, "XGBoost", fold_idx)
    classical_results.append(xgb_res)

    print(f"  Fold {fold_idx+1}  SVM  | F1={svm_res['macro_f1']:.3f} | AUROC={svm_res['auroc']:.3f} | FAR={svm_res['FAR_hr']:.2f}/hr")
    print(f"  Fold {fold_idx+1}  RF   | F1={rf_res['macro_f1']:.3f} | AUROC={rf_res['auroc']:.3f} | FAR={rf_res['FAR_hr']:.2f}/hr")
    print(f"  Fold {fold_idx+1}  XGB  | F1={xgb_res['macro_f1']:.3f} | AUROC={xgb_res['auroc']:.3f} | FAR={xgb_res['FAR_hr']:.2f}/hr")

df_classical = pd.DataFrame(classical_results)
summary_cl = df_classical.groupby("model")[["acc","macro_f1","auroc","FAR_hr"]].agg(["mean","std"])
print("\n[Classical Models – 5-Fold Summary]")
print(summary_cl.round(4).to_string())


  CLASSICAL ML MODELS: SVM · RF · XGBoost  (FFT features)
  Fold 1  SVM  | F1=0.581 | AUROC=0.871 | FAR=34.88/hr
  Fold 1  RF   | F1=0.570 | AUROC=0.825 | FAR=26.89/hr
  Fold 1  XGB  | F1=0.586 | AUROC=0.835 | FAR=31.16/hr
  Fold 2  SVM  | F1=0.608 | AUROC=0.875 | FAR=33.86/hr
  Fold 2  RF   | F1=0.604 | AUROC=0.833 | FAR=22.05/hr
  Fold 2  XGB  | F1=0.605 | AUROC=0.844 | FAR=26.10/hr
  Fold 3  SVM  | F1=0.527 | AUROC=0.801 | FAR=38.14/hr
  Fold 3  RF   | F1=0.551 | AUROC=0.801 | FAR=28.69/hr
  Fold 3  XGB  | F1=0.575 | AUROC=0.811 | FAR=32.51/hr
  Fold 4  SVM  | F1=0.600 | AUROC=0.850 | FAR=29.14/hr
  Fold 4  RF   | F1=0.633 | AUROC=0.860 | FAR=20.70/hr
  Fold 4  XGB  | F1=0.647 | AUROC=0.861 | FAR=22.61/hr
  Fold 5  SVM  | F1=0.599 | AUROC=0.847 | FAR=44.44/hr
  Fold 5  RF   | F1=0.627 | AUROC=0.849 | FAR=30.38/hr
  Fold 5  XGB  | F1=0.623 | AUROC=0.851 | FAR=32.96/hr

[Classical Models – 5-Fold Summary]
            acc         macro_f1           auroc         FAR_hr        
       

In [ ]:
# ── 14. FOCAL LOSS ────────────────────────────────────────────────────────────
if TORCH_AVAILABLE:
    class FocalLoss(nn.Module):
        def __init__(self, gamma: float = FOCAL_GAMMA, class_weights=None, device="cpu"):
            super().__init__()
            self.gamma = gamma
            w = torch.tensor(class_weights, dtype=torch.float32).to(device) \
                if class_weights else None
            self.weight = w
        def forward(self, logits, targets):
            ce  = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
            pt  = torch.exp(-ce)
            return ((1 - pt) ** self.gamma * ce).mean()

In [ ]:
# ── 15. DATASET CLASS ─────────────────────────────────────────────────────────
if TORCH_AVAILABLE:
    class EEGDataset(Dataset):
        def __init__(self, X: np.ndarray, y: np.ndarray):
            self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
            self.y = torch.tensor(y, dtype=torch.long)
        def __len__(self): return len(self.y)
        def __getitem__(self, i): return self.X[i], self.y[i]

    def make_loader(X, y, shuffle=True, sample_weights=None, batch_size=BATCH_SIZE):
        ds = EEGDataset(X, y)
        sampler = None
        if sample_weights is not None:
            w = torch.tensor([sample_weights[yi] for yi in y], dtype=torch.float32)
            sampler = WeightedRandomSampler(w, num_samples=len(w), replacement=True)
        return DataLoader(ds, batch_size=batch_size,
                          sampler=sampler if sampler else None,
                          shuffle=shuffle if sampler is None else False)

In [ ]:
# ── 16. DEEP LEARNING MODEL ARCHITECTURES ────────────────────────────────────
if TORCH_AVAILABLE:
    class CNN(nn.Module):
        def __init__(self, n_classes=4):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm1d(32),
                nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                nn.AdaptiveAvgPool1d(4),
                nn.Flatten(),
                nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(128, n_classes)
            )
        def forward(self, x): return self.net(x)

    class CNNLSTM(nn.Module):
        def __init__(self, n_classes=4):
            super().__init__()
            self.cnn = nn.Sequential(
                nn.Conv1d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            )
            self.lstm = nn.LSTM(64, 64, num_layers=2, batch_first=True,
                                 bidirectional=True, dropout=0.25)
            self.head = nn.Sequential(
                nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(64, n_classes)
            )
        def forward(self, x):
            feat = self.cnn(x).permute(0, 2, 1)
            out, _ = self.lstm(feat)
            return self.head(out[:, -1, :])

    class TransformerEEG(nn.Module):
        def __init__(self, n_classes=4, d_model=64, nhead=4, num_layers=2):
            super().__init__()
            self.embed = nn.Linear(1, d_model)
            enc_layer  = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                     dim_feedforward=128, dropout=0.1,
                                                     batch_first=True, norm_first=True)
            self.enc   = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
            self.head  = nn.Sequential(
                nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(0.2),
                nn.Linear(64, n_classes)
            )
        def forward(self, x):
            seq = x.permute(0, 2, 1)
            emb = self.embed(seq)
            ctx = self.enc(emb)
            return self.head(ctx.mean(dim=1))

    class BiMambaBlock(nn.Module):
        """
        GRU-based approximation of selective state-space dynamics.
        Processes forward and backward with GRU gating and residual connection.
        Full Mamba SSM requires mamba-ssm; this serves as the implementation
        when that library is unavailable.
        """
        def __init__(self, d_model: int = 128):
            super().__init__()
            self.fwd = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=False)
            self.bwd = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=False)
            self.gate   = nn.Linear(d_model, d_model)
            self.norm   = nn.LayerNorm(d_model)
            self.proj   = nn.Linear(d_model, d_model)
        def forward(self, x):
            f_out, _ = self.fwd(x)
            b_out, _ = self.bwd(x.flip(1))
            merged   = torch.cat([f_out, b_out.flip(1)], dim=-1)
            gate     = torch.sigmoid(self.gate(merged))
            gated    = gate * merged
            return self.norm(x + self.proj(gated))

    class BoundaryDisc(nn.Module):
        def __init__(self, d_model: int = 256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d_model, 128), nn.LeakyReLU(0.2),
                nn.Linear(128, 64),     nn.LeakyReLU(0.2),
                nn.Linear(64, 1)
            )
        def forward(self, x): return self.net(x)

    class BiMambaSTAN(nn.Module):
        def __init__(self, n_classes=4, d_model=128):
            super().__init__()
            self.input_proj = nn.Linear(16, d_model)
            self.temp_enc = nn.Sequential(*[BiMambaBlock(d_model) for _ in range(3)])
            self.spatial_enc = nn.Sequential(
                nn.Linear(16, d_model), nn.GELU(),
                nn.MultiheadAttention(d_model, num_heads=4, batch_first=True, dropout=0.1),
            )
            self.fuse  = nn.Linear(d_model * 2, 256)
            self.head  = nn.Sequential(
                nn.GELU(),
                nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(128, n_classes)
            )
            self.disc  = BoundaryDisc(256)
        def encode(self, x):
            feat = x.squeeze(1)
            t = self.input_proj(feat).unsqueeze(1)
            t = t.expand(-1, 16, -1)
            for block in self.temp_enc:
                t = block(t)
            t_out = t.mean(dim=1)
            s = nn.functional.gelu(nn.Linear(16, 128, device=feat.device)(feat))
            z = self.fuse(torch.cat([t_out, s], dim=-1))
            return z
        def forward(self, x):
            z = self.encode(x)
            return self.head(z), z

    MODEL_REGISTRY = {
        "CNN":          CNN,
        "CNN-LSTM":     CNNLSTM,
        "Transformer":  TransformerEEG,
        "BiMamba-STAN": BiMambaSTAN,
    }

    def train_epoch(model, loader, optimizer, criterion, device, model_name,
                    adv_optimizer=None, disc=None):
        model.train()
        total_loss = 0
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(X_b)
            if isinstance(out, tuple):
                logits, emb = out
                main_loss   = criterion(logits, y_b)
                mask_03 = (y_b == 0) | (y_b == 3)
                if mask_03.sum() > 1 and disc is not None:
                    real_mask = (y_b[mask_03] == 3).float().unsqueeze(1)
                    d_out     = disc(emb[mask_03].detach())
                    adv_loss  = -torch.mean(d_out * (2 * real_mask - 1))
                    loss      = main_loss + ADV_WEIGHT * adv_loss
                else:
                    loss = main_loss
            else:
                logits = out
                loss   = criterion(logits, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)

    @torch.no_grad()
    def evaluate_epoch(model, loader, device):
        model.eval()
        preds, probs_all, labels = [], [], []
        for X_b, y_b in loader:
            X_b = X_b.to(device)
            out = model(X_b)
            logits = out[0] if isinstance(out, tuple) else out
            p = torch.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(p)
            preds.extend(logits.argmax(1).cpu().numpy())
            labels.extend(y_b.numpy())
        probs_all = np.vstack(probs_all)
        preds     = np.array(preds)
        labels    = np.array(labels)
        try:
            auc = roc_auc_score(label_binarize(labels, classes=[0,1,2,3]),
                                 probs_all, multi_class="ovr", average="macro")
        except Exception:
            auc = 0.0
        return preds, probs_all, labels, auc

    def train_model(model_cls, fd, device, model_name):
        set_seed()
        criterion = FocalLoss(gamma=FOCAL_GAMMA, class_weights=CLASS_WEIGHTS, device=device)
        if model_name == "BiMamba-STAN":
            model = model_cls().to(device)
            disc  = BoundaryDisc(256).to(device)
            opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            adv_opt = torch.optim.AdamW(disc.parameters(), lr=LR, weight_decay=WD)
        else:
            model  = model_cls().to(device)
            disc   = None
            opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
            adv_opt = None
        tr_loader = make_loader(fd["X_tr"], fd["y_tr"], sample_weights=SAMPLE_W)
        va_loader = make_loader(fd["X_va"], fd["y_va"], shuffle=False)
        best_auc, best_state, no_improve = 0.0, None, 0
        history = {"train_loss": [], "val_auc": []}
        for epoch in range(MAX_EPOCHS):
            loss = train_epoch(model, tr_loader, opt, criterion, device, model_name, adv_opt, disc)
            _, _, _, val_auc = evaluate_epoch(model, va_loader, device)
            history["train_loss"].append(loss)
            history["val_auc"].append(val_auc)
            if val_auc > best_auc:
                best_auc   = val_auc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                break
        model.load_state_dict(best_state)
        return model, history, best_auc

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[DL] Training device: {device}")

    dl_results = {mname: [] for mname in MODEL_REGISTRY}
    dl_predictions = {mname: {"y_true": [], "y_pred": [], "y_prob": []}
                      for mname in MODEL_REGISTRY}

    for mname, mcls in MODEL_REGISTRY.items():
        print(f"\n  Training {mname}...")
        for fold_idx, fd in enumerate(fold_data):
            t0 = time.time()
            model, history, best_auc = train_model(mcls, fd, device, mname)
            elapsed = time.time() - t0
            va_loader = make_loader(fd["X_va"], fd["y_va"], shuffle=False)
            preds, probs, labels, auc = evaluate_epoch(model, va_loader, device)
            far = compute_far(labels, preds, len(labels))
            row = {
                "model": mname, "fold": fold_idx + 1,
                "acc":      accuracy_score(labels, preds),
                "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
                "auroc":    auc, "FAR_hr": far,
                "train_time_s": elapsed,
            }
            dl_results[mname].append(row)
            dl_predictions[mname]["y_true"].extend(labels.tolist())
            dl_predictions[mname]["y_pred"].extend(preds.tolist())
            dl_predictions[mname]["y_prob"].extend(probs.tolist())
            print(f"    Fold {fold_idx+1}: F1={row['macro_f1']:.3f} | "
                  f"AUROC={auc:.3f} | FAR={far:.2f}/hr | {elapsed:.0f}s")

    df_dl = pd.DataFrame([r for rows in dl_results.values() for r in rows])
    print("\n[DL Models – 5-Fold Summary]")
    dl_summary = df_dl.groupby("model")[["acc","macro_f1","auroc","FAR_hr"]].agg(["mean","std"])
    print(dl_summary.round(4).to_string())


[DL] Training device: cuda

  Training CNN...
    Fold 1: F1=0.794 | AUROC=0.937 | FAR=18.90/hr | 25s
    Fold 2: F1=0.821 | AUROC=0.958 | FAR=11.70/hr | 11s
    Fold 3: F1=0.685 | AUROC=0.877 | FAR=16.31/hr | 7s
    Fold 4: F1=0.804 | AUROC=0.948 | FAR=10.12/hr | 10s
    Fold 5: F1=0.776 | AUROC=0.938 | FAR=18.79/hr | 10s

  Training CNN-LSTM...
    Fold 1: F1=0.650 | AUROC=0.896 | FAR=26.66/hr | 8s
    Fold 2: F1=0.750 | AUROC=0.922 | FAR=17.32/hr | 16s
    Fold 3: F1=0.668 | AUROC=0.874 | FAR=19.91/hr | 27s
    Fold 4: F1=0.754 | AUROC=0.916 | FAR=16.88/hr | 20s
    Fold 5: F1=0.709 | AUROC=0.903 | FAR=22.61/hr | 23s

  Training Transformer...
    Fold 1: F1=0.501 | AUROC=0.858 | FAR=35.66/hr | 8s
    Fold 2: F1=0.522 | AUROC=0.855 | FAR=32.29/hr | 16s
    Fold 3: F1=0.467 | AUROC=0.791 | FAR=31.39/hr | 32s
    Fold 4: F1=0.604 | AUROC=0.865 | FAR=25.20/hr | 20s
    Fold 5: F1=0.525 | AUROC=0.854 | FAR=32.85/hr | 36s

  Training BiMamba-STAN...
    Fold 1: F1=0.791 | AUROC=0.945 | 

In [ ]:
# ── 17. FALSE ALARM ANALYSIS ──────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  FALSE ALARM ANALYSIS")
print("=" * 70)

classical_preds = {m: {"y_true": [], "y_pred": [], "y_prob": []}
                   for m in ["SVM", "RF", "XGBoost"]}
for fold_idx, fd in enumerate(fold_data):
    scaler_fa = StandardScaler()
    X_tr_s = scaler_fa.fit_transform(fd["X_tr_fft"])
    X_va_s = scaler_fa.transform(fd["X_va_fft"])
    for model_name, model_obj in [("SVM", svm), ("RF", rf), ("XGBoost", xgb_clf)]:
        pred = model_obj.predict(X_va_s)
        prob = model_obj.predict_proba(X_va_s)
        classical_preds[model_name]["y_true"].extend(fd["y_va"])
        classical_preds[model_name]["y_pred"].extend(pred)
        classical_preds[model_name]["y_prob"].extend(prob)
for m in classical_preds:
    for k in classical_preds[m]:
        classical_preds[m][k] = np.array(classical_preds[m][k])

all_model_preds = {}
for m in ["SVM", "RF", "XGBoost"]:
    all_model_preds[m] = classical_preds[m]
if TORCH_AVAILABLE:
    for m in dl_predictions:
        all_model_preds[m] = {
            "y_true": np.array(dl_predictions[m]["y_true"]),
            "y_pred": np.array(dl_predictions[m]["y_pred"]),
            "y_prob": np.array(dl_predictions[m]["y_prob"]),
        }

far_summary = []
for mname, data in all_model_preds.items():
    yt = data["y_true"]; yp = data["y_pred"]
    n_segs = len(yt)
    monitoring_hours = (n_segs * WIN_DURATION) / 3600.0
    FP = int(np.sum((yp == 2) & (yt != 2)))
    TP = int(np.sum((yp == 2) & (yt == 2)))
    FN = int(np.sum((yp != 2) & (yt == 2)))
    TN = int(np.sum((yp != 2) & (yt != 2)))
    far_hr    = FP / monitoring_hours if monitoring_hours > 0 else np.nan
    precision = TP / (TP + FP + 1e-9)
    recall    = TP / (TP + FN + 1e-9)
    mask_03   = (yt == 0) | (yt == 3)
    c0_as_c3  = int(np.sum((yp[mask_03] == 3) & (yt[mask_03] == 0)))
    c3_as_c0  = int(np.sum((yp[mask_03] == 0) & (yt[mask_03] == 3)))
    far_summary.append({
        "Model": mname, "FP (ictal)": FP, "TP (ictal)": TP, "FN (ictal)": FN,
        "Monitoring hrs": round(monitoring_hours, 2),
        "FAR/hr": round(far_hr, 3), "Seizure Precision": round(precision, 3),
        "Seizure Recall": round(recall, 3),
        "C0→C3 confusions": c0_as_c3, "C3→C0 confusions": c3_as_c0,
    })

df_far = pd.DataFrame(far_summary).set_index("Model")
print("\n[FAR Summary Table]")
print(df_far.to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("False Alarm Analysis", fontsize=13, fontweight="bold")
model_names_plot = list(df_far.index)
colors_plot = plt.cm.tab10(np.linspace(0, 0.7, len(model_names_plot)))
axes[0].barh(model_names_plot, df_far["FAR/hr"], color=colors_plot, edgecolor="k", lw=0.7)
axes[0].axvline(FAR_THRESHOLD, color="red", lw=2, ls="--", label=f"Clinical limit ({FAR_THRESHOLD}/hr)")
axes[0].set_xlabel("False Alarm Rate / hour")
axes[0].set_title("FAR/hr per Model")
axes[0].legend(fontsize=8)
axes[0].invert_yaxis()
axes[1].barh(model_names_plot, df_far["FP (ictal)"], color=colors_plot, edgecolor="k", lw=0.7)
axes[1].set_xlabel("Total False Positives (ictal)")
axes[1].set_title("False Positives (predicted seizure ≠ actual)")
axes[1].invert_yaxis()
x = np.arange(len(model_names_plot)); w = 0.35
axes[2].bar(x - w/2, df_far["C0→C3 confusions"], w, label="Normal→Artefact", color="#E53935", alpha=0.8, edgecolor="k")
axes[2].bar(x + w/2, df_far["C3→C0 confusions"], w, label="Artefact→Normal", color="#3949AB", alpha=0.8, edgecolor="k")
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_names_plot, rotation=25, ha="right", fontsize=8)
axes[2].set_ylabel("Count")
axes[2].set_title("Class 0 ↔ Class 3 Confusion")
axes[2].legend(fontsize=8)
plt.tight_layout()
save_fig("EXT_09_false_alarm_analysis")


  FALSE ALARM ANALYSIS

[FAR Summary Table]
              FP (ictal)  TP (ictal)  FN (ictal)  Monitoring hrs  FAR/hr  Seizure Precision  Seizure Recall  C0→C3 confusions  C3→C0 confusions
Model                                                                                                                                          
SVM                 1627        1634         366           44.44  36.608              0.501           0.817                16                 4
RF                   907         996        1004           44.44  20.408              0.523           0.498                 2                 5
XGBoost              839         944        1056           44.44  18.878              0.529           0.472                 5                12
CNN                  674        1511         489           44.44  15.165              0.692           0.755                 8                 0
CNN-LSTM             919        1504         496           44.44  20.678              0.621

PosixPath('/content/outputs/EXT_09_false_alarm_analysis.png')

In [ ]:
# ── 18. MODEL EVALUATION ──────────────────────────────────────────────────────
print("=" * 70)
print("  MODEL EVALUATION – CONFUSION MATRICES & ROC CURVES")
print("=" * 70)

eval_rows = []
for mname, data in all_model_preds.items():
    yt = data["y_true"]; yp = data["y_pred"]; yprob = data["y_prob"]
    n_segs = len(yt)
    acc  = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, average="macro", zero_division=0)
    rec  = recall_score(yt, yp, average="macro", zero_division=0)
    f1   = f1_score(yt, yp, average="macro", zero_division=0)
    far  = compute_far(yt, yp, n_segs)
    try:
        auc = roc_auc_score(label_binarize(yt, classes=[0,1,2,3]),
                             yprob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    eval_rows.append({"Model": mname, "Accuracy": round(acc, 4),
                      "Precision": round(prec, 4), "Recall": round(rec, 4),
                      "Macro F1": round(f1, 4), "AUROC": round(auc, 4),
                      "FAR/hr": round(far, 3),
                      "FAR ✓": "✓" if far <= FAR_THRESHOLD else "✗",
                      "F1 ✓":  "✓" if f1  >= 0.90         else "✗"})

df_eval = pd.DataFrame(eval_rows).set_index("Model")
print("\n[Full Evaluation Table]")
print(df_eval.to_string())

n_models = len(all_model_preds)
ncols = min(4, n_models)
nrows = int(np.ceil(n_models / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = np.array(axes).flatten()
fig.suptitle("Normalised Confusion Matrices (All Folds Combined)", fontsize=13, fontweight="bold")
for i, (mname, data) in enumerate(all_model_preds.items()):
    cm = confusion_matrix(data["y_true"], data["y_pred"], normalize="true")
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues", values_format=".2f")
    axes[i].set_title(mname, fontsize=10)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
save_fig("EXT_10a_confusion_matrices_all_models")

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_title("ROC Curves – Seizure Detection (Class 2, One-vs-Rest)", fontsize=12, fontweight="bold")
for i, (mname, data) in enumerate(all_model_preds.items()):
    yt_bin = (data["y_true"] == 2).astype(int)
    yprob  = np.array(data["y_prob"])
    if yprob.ndim == 2 and yprob.shape[1] >= 3:
        score = yprob[:, 2]
    else:
        score = yprob[:, 0] if yprob.ndim == 2 else yprob
    try:
        fpr, tpr, _ = roc_curve(yt_bin, score)
        auc_cls2 = roc_auc_score(yt_bin, score)
        ax.plot(fpr, tpr, lw=2, color=f"C{i}", label=f"{mname}  (AUC={auc_cls2:.3f})")
    except Exception:
        pass
ax.plot([0,1],[0,1],"k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig("EXT_10b_roc_curves_seizure")
print("[EVAL] Evaluation figures saved.\n")

  MODEL EVALUATION – CONFUSION MATRICES & ROC CURVES

[Full Evaluation Table]
              Accuracy  Precision  Recall  Macro F1   AUROC  FAR/hr FAR ✓ F1 ✓
Model                                                                         
SVM             0.6514     0.6563  0.6514    0.6040  0.8617  36.608     ✗    ✗
RF              0.6229     0.6239  0.6229    0.6232  0.8511  20.408     ✗    ✗
XGBoost         0.6214     0.6198  0.6214    0.6196  0.8520  18.878     ✗    ✗
CNN             0.7759     0.7786  0.7759    0.7765  0.9308  15.165     ✗    ✗
CNN-LSTM        0.7111     0.7190  0.7111    0.7105  0.8916  20.678     ✗    ✗
Transformer     0.5710     0.5713  0.5710    0.5276  0.8379  31.478     ✗    ✗
BiMamba-STAN    0.8043     0.8045  0.8043    0.8039  0.9330  13.500     ✗    ✗
[SAVED] /content/outputs/EXT_10a_confusion_matrices_all_models.png
[SAVED] /content/outputs/EXT_10b_roc_curves_seizure.png
[EVAL] Evaluation figures saved.



In [ ]:
# ── 19. REPRESENTATION COMPARISON (Architecture-controlled MLP) ───────────────
print("=" * 70)
print("  REPRESENTATION COMPARISON (Objective 4)")
print("=" * 70)

OBJECTIVE_IMPROVEMENT = 10.0

class MLPBaseline(nn.Module):
    def __init__(self, n_in=16, n_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 64),  nn.GELU(), nn.BatchNorm1d(64),  nn.Dropout(0.3),
            nn.Linear(64,  128),  nn.GELU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64),   nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x.squeeze(1))

def train_mlp(X_tr, y_tr, X_va, y_va, n_in, device):
    set_seed()
    model = MLPBaseline(n_in=n_in).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit  = FocalLoss(gamma=FOCAL_GAMMA, class_weights=CLASS_WEIGHTS, device=device)
    tr_ld = make_loader(X_tr, y_tr, sample_weights=SAMPLE_W)
    va_ld = make_loader(X_va, y_va, shuffle=False)
    best_auc, best_state, no_imp = 0.0, None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for Xb, yb in tr_ld:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(Xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        preds_v, probs_v, labels_v, auc_v = evaluate_epoch(model, va_ld, device)
        sched.step(1 - auc_v)
        if auc_v > best_auc:
            best_auc   = auc_v
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break
    model.load_state_dict(best_state)
    preds_v, probs_v, labels_v, auc_v = evaluate_epoch(model, va_ld, device)
    f1_v = f1_score(labels_v, preds_v, average="macro", zero_division=0)
    return f1_v, auc_v

rep_raw, rep_fft = [], []
print("\n  Training architecture-matched MLP on Raw and FFT features (5 folds)...")
for fold_idx, fd in enumerate(fold_data):
    f1_r, auc_r = train_mlp(fd["X_tr"], fd["y_tr"], fd["X_va"], fd["y_va"], n_in=16, device=device)
    f1_f, auc_f = train_mlp(fd["X_tr_fft"], fd["y_tr"], fd["X_va_fft"], fd["y_va"], n_in=5, device=device)
    rep_raw.append({"fold": fold_idx+1, "f1": f1_r, "auc": auc_r})
    rep_fft.append({"fold": fold_idx+1, "f1": f1_f, "auc": auc_f})
    print(f"  Fold {fold_idx+1}: Raw F1={f1_r:.3f} AUC={auc_r:.3f}  |  FFT F1={f1_f:.3f} AUC={auc_f:.3f}")

raw_f1s  = np.array([r["f1"]  for r in rep_raw])
raw_aucs = np.array([r["auc"] for r in rep_raw])
fft_f1s  = np.array([r["f1"]  for r in rep_fft])
fft_aucs = np.array([r["auc"] for r in rep_fft])
delta_f1  = (raw_f1s  - fft_f1s)  / (fft_f1s  + 1e-9) * 100
delta_auc = (raw_aucs - fft_aucs) / (fft_aucs + 1e-9) * 100
rep_pass  = delta_f1.mean() >= OBJECTIVE_IMPROVEMENT or delta_auc.mean() >= OBJECTIVE_IMPROVEMENT
print(f"\n  Raw  F1: {raw_f1s.mean():.4f} ± {raw_f1s.std():.4f}")
print(f"  FFT  F1: {fft_f1s.mean():.4f} ± {fft_f1s.std():.4f}")
print(f"  Δ F1:    {delta_f1.mean():+.2f}%  {'✓' if delta_f1.mean() >= OBJECTIVE_IMPROVEMENT else '✗'}")
print(f"\n  Raw  AUROC: {raw_aucs.mean():.4f} ± {raw_aucs.std():.4f}")
print(f"  FFT  AUROC: {fft_aucs.mean():.4f} ± {fft_aucs.std():.4f}")
print(f"  Δ AUROC:    {delta_auc.mean():+.2f}%  {'✓' if delta_auc.mean() >= OBJECTIVE_IMPROVEMENT else '✗'}")
print(f"\n  Objective 4 overall: {'✓ MET' if rep_pass else '✗ NOT MET'}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Representation Comparison: Raw Temporal vs FFT Spectral\n"
             "(Architecture-controlled MLP — only input changes)", fontsize=12, fontweight="bold")
folds = np.arange(1, N_FOLDS + 1)
for ax, (raw_arr, fft_arr, metric) in zip(axes, [
    (raw_f1s,  fft_f1s,  "Macro F1"),
    (raw_aucs, fft_aucs, "AUROC"),
]):
    ax.plot(folds, raw_arr, "o-",  lw=2, color="#1565C0", label=f"Raw Temporal  (μ={raw_arr.mean():.4f})")
    ax.plot(folds, fft_arr, "s--", lw=2, color="#E65100", label=f"FFT Spectral  (μ={fft_arr.mean():.4f})")
    ax.fill_between(folds, raw_arr, fft_arr, alpha=0.12, color="#607D8B")
    mean_delta_pct = (raw_arr - fft_arr).mean() / fft_arr.mean() * 100
    ax.set_xlabel("Fold"); ax.set_ylabel(metric)
    ax.set_title(f"{metric} — Mean Δ = {mean_delta_pct:+.2f}%\n(objective ≥ {OBJECTIVE_IMPROVEMENT}%)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.set_xticks(folds)
plt.tight_layout()
save_fig("FIX_03_representation_comparison")


  REPRESENTATION COMPARISON (Objective 4)

  Training architecture-matched MLP on Raw and FFT features (5 folds)...
  Fold 1: Raw F1=0.703 AUC=0.936  |  FFT F1=0.545 AUC=0.855
  Fold 2: Raw F1=0.771 AUC=0.959  |  FFT F1=0.524 AUC=0.874
  Fold 3: Raw F1=0.720 AUC=0.912  |  FFT F1=0.506 AUC=0.796
  Fold 4: Raw F1=0.749 AUC=0.940  |  FFT F1=0.568 AUC=0.885
  Fold 5: Raw F1=0.732 AUC=0.929  |  FFT F1=0.564 AUC=0.849

  Raw  F1: 0.7352 ± 0.0236
  FFT  F1: 0.5413 ± 0.0235
  Δ F1:    +36.08%  ✓

  Raw  AUROC: 0.9350 ± 0.0152
  FFT  AUROC: 0.8519 ± 0.0309
  Δ AUROC:    +9.85%  ✗

  Objective 4 overall: ✓ MET
[SAVED] /content/outputs/FIX_03_representation_comparison.png


PosixPath('/content/outputs/FIX_03_representation_comparison.png')

In [ ]:
# ── 20. MODEL SELECTION ───────────────────────────────────────────────────────
print("=" * 70)
print("  MODEL SELECTION")
print("=" * 70)

df_rank = df_eval[["Macro F1", "AUROC", "FAR/hr"]].copy()
df_rank["FAR_rank"] = df_rank["FAR/hr"].rank(ascending=True)
df_rank["F1_rank"]  = df_rank["Macro F1"].rank(ascending=False)
df_rank["AUC_rank"] = df_rank["AUROC"].rank(ascending=False)
df_rank["Combined_rank"] = df_rank[["FAR_rank","F1_rank","AUC_rank"]].mean(axis=1)
df_rank = df_rank.sort_values("Combined_rank")
df_rank["Rank"] = range(1, len(df_rank)+1)
print("\n[Model Rankings]")
print(df_rank[["Rank","Macro F1","AUROC","FAR/hr","Combined_rank"]].to_string())
top3 = df_rank.index[:3].tolist()
print(f"\n  ★ Top 3 models: {top3}")

metrics_radar = ["Macro F1", "AUROC", "Accuracy", "Seizure Recall", "Seizure Precision"]
df_eval_ext = df_eval.copy()
df_eval_ext["Seizure Recall"]    = df_far["Seizure Recall"]
df_eval_ext["Seizure Precision"] = df_far["Seizure Precision"]
fig = plt.figure(figsize=(15, 5))
fig.suptitle("Model Selection – Top 3 Model Radar Profiles", fontsize=13, fontweight="bold")
for plot_i, mname in enumerate(top3):
    ax = fig.add_subplot(1, 3, plot_i+1, projection="polar")
    row = df_eval_ext.loc[mname]
    vals = [row["Macro F1"], row["AUROC"], row["Accuracy"],
            row.get("Seizure Recall", 0), row.get("Seizure Precision", 0)]
    vals_norm = [min(v, 1.0) for v in vals]
    N = len(metrics_radar)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    vals_norm += vals_norm[:1]; angles += angles[:1]
    ax.plot(angles, vals_norm, color=f"C{plot_i}", lw=2)
    ax.fill(angles, vals_norm, color=f"C{plot_i}", alpha=0.2)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_radar, fontsize=7)
    ax.set_ylim(0, 1); ax.set_title(f"#{plot_i+1} {mname}", fontsize=10, pad=14)
plt.tight_layout()
save_fig("EXT_11_model_selection_radar")
print("[MODEL SELECTION] Complete.\n")

  MODEL SELECTION

[Model Rankings]
              Rank  Macro F1   AUROC  FAR/hr  Combined_rank
Model                                                      
BiMamba-STAN     1    0.8039  0.9330  13.500       1.000000
CNN              2    0.7765  0.9308  15.165       2.000000
CNN-LSTM         3    0.7105  0.8916  20.678       3.666667
XGBoost          4    0.6196  0.8520  18.878       4.333333
RF               5    0.6232  0.8511  20.408       4.666667
SVM              6    0.6040  0.8617  36.608       5.666667
Transformer      7    0.5276  0.8379  31.478       6.666667

  ★ Top 3 models: ['BiMamba-STAN', 'CNN', 'CNN-LSTM']
[SAVED] /content/outputs/EXT_11_model_selection_radar.png
[MODEL SELECTION] Complete.



In [ ]:
# ── 21. INTERPRETABILITY ──────────────────────────────────────────────────────
print("=" * 70)
print("  INTERPRETABILITY ANALYSIS — TOP 3 MODELS")
print("=" * 70)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed()
fd5       = fold_data[4]
scaler12  = StandardScaler()
X_tr5_raw = scaler12.fit_transform(fd5["X_tr"])
X_va5_raw = scaler12.transform(fd5["X_va"])
y_tr5     = fd5["y_tr"]
y_va5     = fd5["y_va"]
N_RAW_FEATURES = X_tr5_raw.shape[1]
n_classes_shap = len(np.unique(y_tr5))
ch_labels_shap = [f"X{i+1}" for i in range(N_RAW_FEATURES)]
print(f"  Raw features: {N_RAW_FEATURES}  |  Classes: {n_classes_shap}  |  Device: {DEVICE}")

print("\n  Re-training top-3 models on fold 5 for SHAP...")
top3_model_objects = {}
for mname in top3:
    if mname not in MODEL_REGISTRY:
        print(f"  {mname} not in MODEL_REGISTRY — skipping.")
        continue
    print(f"    {mname} ...", end=" ", flush=True)
    model_fold5, _, _ = train_model(MODEL_REGISTRY[mname], fd5, DEVICE, mname)
    model_fold5.eval()
    top3_model_objects[mname] = model_fold5
    print("done.")
print(f"\n  Models ready: {list(top3_model_objects.keys())}")

def dl_predict_proba(model, X_np, device):
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1).to(device)
    with torch.no_grad():
        out = model(X_t)
        logits = out[0] if isinstance(out, tuple) else out
    return torch.softmax(logits, dim=-1).cpu().numpy()

def run_kernel_shap(predict_fn, X_val, nsamples=80):
    n_bg  = min(50, len(X_val))
    n_exp = min(100, len(X_val))
    rng   = np.random.default_rng(SEED)
    bg    = X_val[rng.choice(len(X_val), n_bg,  replace=False)]
    exp   = X_val[rng.choice(len(X_val), n_exp, replace=False)]
    expl  = shap.KernelExplainer(predict_fn, bg, link="identity")
    sv    = expl.shap_values(exp, nsamples=nsamples)
    arr   = np.stack([np.array(s) for s in sv], axis=0)
    return arr, exp

shap_results = {}
for mname, model in top3_model_objects.items():
    print(f"\n[SHAP] KernelSHAP → {mname} ...")
    predict_fn = lambda X_np, m=model: dl_predict_proba(m, X_np, DEVICE)
    shap_arr, X_exp = run_kernel_shap(predict_fn, X_va5_raw, nsamples=80)
    global_imp  = np.mean(np.abs(shap_arr).mean(axis=1), axis=0)
    ch_rank     = np.argsort(global_imp)[::-1]
    shap_results[mname] = {
        "shap_arr":     shap_arr, "X_exp": X_exp,
        "global_imp":   global_imp, "ch_rank": ch_rank,
        "sorted_names": [ch_labels_shap[i] for i in ch_rank],
        "sorted_vals":  global_imp[ch_rank],
    }
    print(f"  Top channels: {[ch_labels_shap[i] for i in ch_rank[:min(5,N_RAW_FEATURES)]]}")

n_m = len(shap_results)
fig, axes = plt.subplots(1, n_m, figsize=(6*n_m, max(5, N_RAW_FEATURES*0.5+2)), sharey=False)
axes = [axes] if n_m == 1 else list(axes)
fig.suptitle("Global SHAP Channel Importance — Top 3 Models", fontsize=13, fontweight="bold")
for ax, (mname, res) in zip(axes, shap_results.items()):
    cols = ["#B71C1C" if i < min(5,N_RAW_FEATURES) else "#37474F" for i in range(N_RAW_FEATURES)]
    ax.barh(res["sorted_names"], res["sorted_vals"], color=cols, edgecolor="k", lw=0.5)
    ax.set_xlabel("|SHAP| (mean)"); ax.set_title(mname, fontsize=11); ax.invert_yaxis()
    mx = max(res["sorted_vals"])
    for j, (nm, vl) in enumerate(zip(res["sorted_names"], res["sorted_vals"])):
        ax.text(vl + mx*0.01, j, f"{vl:.4f}", va="center", fontsize=7)
plt.tight_layout()
save_fig("EXT_12a_shap_global_importance_top3")

fig, axes = plt.subplots(1, n_m, figsize=(6*n_m, max(4, n_classes_shap+2)))
axes = [axes] if n_m == 1 else list(axes)
fig.suptitle("Per-Class SHAP Heatmap — Top 3 Models", fontsize=13, fontweight="bold")
for ax, (mname, res) in zip(axes, shap_results.items()):
    per_cls    = np.abs(res["shap_arr"]).mean(axis=1)
    n_feats_actual = per_cls.shape[1]
    ch_rank_actual = res["ch_rank"][:n_feats_actual]
    names_actual   = [ch_labels_shap[i] for i in ch_rank_actual]
    im = ax.imshow(per_cls[:, ch_rank_actual], aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(n_feats_actual))
    ax.set_xticklabels(names_actual, rotation=90, fontsize=7)
    ax.set_yticks(range(n_classes_shap))
    ax.set_yticklabels(CLASS_NAMES[:n_classes_shap], fontsize=8)
    ax.set_title(mname, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, label="|SHAP|")
plt.tight_layout()
save_fig("EXT_12b_shap_heatmap_top3")

_cmap = _cm.get_cmap("coolwarm")
for mname, res in shap_results.items():
    raw_cls0   = np.array(res["shap_arr"][0])
    if raw_cls0.shape[0] < raw_cls0.shape[1]:
        shap_fixed = [np.array(s).T for s in res["shap_arr"]]
    else:
        shap_fixed = [np.array(s) for s in res["shap_arr"]]
    n_samples      = shap_fixed[0].shape[0]
    n_feats_actual = shap_fixed[0].shape[1]
    ch_rank_actual = res["ch_rank"][:n_feats_actual]
    names_actual   = [ch_labels_shap[i] for i in ch_rank_actual]
    n_classes_plot = len(shap_fixed)
    X_exp = res["X_exp"]
    if X_exp.shape[0] != n_samples:
        X_exp = X_exp.T
    fig, axes = plt.subplots(1, n_classes_plot, figsize=(4.5 * n_classes_plot, max(5, n_feats_actual * 0.8 + 2)))
    axes = [axes] if n_classes_plot == 1 else list(axes)
    fig.suptitle(f"SHAP Dot Plot — {mname}", fontsize=12, fontweight="bold")
    rng_s = np.random.default_rng(SEED)
    for cls, ax in enumerate(axes):
        sv = shap_fixed[cls][:, ch_rank_actual]
        fv = X_exp[:, ch_rank_actual]
        for fi in range(n_feats_actual):
            col_sv = sv[:, fi]
            col_fv = fv[:, fi]
            lo, hi = col_fv.min(), col_fv.max()
            sc_norm = (col_fv - lo) / (hi - lo + 1e-9)
            dot_colors = [_cmap(float(v))[:3] for v in sc_norm]
            jit   = rng_s.uniform(-0.25, 0.25, n_samples)
            y_pos = np.full(n_samples, float(fi)) + jit
            ax.scatter(col_sv, y_pos, s=8, alpha=0.5, color=dot_colors, edgecolors="none", linewidths=0)
        ax.set_yticks(range(n_feats_actual))
        ax.set_yticklabels(names_actual, fontsize=7)
        ax.set_xlabel("SHAP value", fontsize=8)
        ax.axvline(0, color="k", lw=0.8, ls="--")
        ax.set_title(CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"Class {cls}", fontsize=9)
        ax.grid(axis="x", alpha=0.2)
    _norm = _mc.Normalize(vmin=0, vmax=1)
    _sm   = _cm.ScalarMappable(norm=_norm, cmap=_cmap)
    _sm.set_array([])
    fig.colorbar(_sm, ax=axes, label="Feature value (low→high)", shrink=0.55, pad=0.02)
    plt.tight_layout()
    save_fig(f"EXT_12c_dot_{mname.replace('-','_').replace(' ','_')}")

_ref_model = top3[0] if top3[0] in shap_results else list(shap_results.keys())[0]
_ref_sv0   = np.array(shap_results[_ref_model]["shap_arr"][0])
if _ref_sv0.shape[0] < _ref_sv0.shape[1]:
    _ref_sv0 = _ref_sv0.T
_n_feats_ref = _ref_sv0.shape[1]
_rank_ref    = shap_results[_ref_model]["ch_rank"][:_n_feats_ref]
top5_ch      = [ch_labels_shap[i] for i in _rank_ref[:min(5, _n_feats_ref)]]
sorted_names = [ch_labels_shap[i] for i in _rank_ref]
sorted_vals  = shap_results[_ref_model]["global_imp"][_rank_ref]
print(f"\n[INTERPRETABILITY] Complete. Reference model: {_ref_model}. Top channels: {top5_ch}\n")

  INTERPRETABILITY ANALYSIS — TOP 3 MODELS
  Raw features: 16  |  Classes: 4  |  Device: cuda

  Re-training top-3 models on fold 5 for SHAP...
    BiMamba-STAN ... done.
    CNN ... done.
    CNN-LSTM ... done.

  Models ready: ['BiMamba-STAN', 'CNN', 'CNN-LSTM']

[SHAP] KernelSHAP → BiMamba-STAN ...


  0%|          | 0/100 [00:00<?, ?it/s]

  Top channels: ['X2', 'X4', 'X3', 'X1']

[SHAP] KernelSHAP → CNN ...


  0%|          | 0/100 [00:00<?, ?it/s]

  Top channels: ['X4', 'X2', 'X3', 'X1']

[SHAP] KernelSHAP → CNN-LSTM ...


  0%|          | 0/100 [00:00<?, ?it/s]

  Top channels: ['X4', 'X2', 'X3', 'X1']
[SAVED] /content/outputs/EXT_12a_shap_global_importance_top3.png
[SAVED] /content/outputs/EXT_12b_shap_heatmap_top3.png
[SAVED] /content/outputs/EXT_12c_dot_BiMamba_STAN.png
[SAVED] /content/outputs/EXT_12c_dot_CNN.png
[SAVED] /content/outputs/EXT_12c_dot_CNN_LSTM.png

[INTERPRETABILITY] Complete. Reference model: BiMamba-STAN. Top channels: ['X2', 'X4', 'X3', 'X1']



In [ ]:
# ── 22. SHAP STABILITY ACROSS FOLDS ──────────────────────────────────────────
print("=" * 70)
print("  SHAP ATTRIBUTION STABILITY ACROSS FOLDS")
print("=" * 70)

def run_shap_fold(model, X_val, device, nsamples=60, n_bg=40, n_exp=80):
    rng = np.random.default_rng(SEED)
    bg  = X_val[rng.choice(len(X_val), min(n_bg,  len(X_val)), replace=False)]
    exp = X_val[rng.choice(len(X_val), min(n_exp, len(X_val)), replace=False)]
    pred_fn = lambda Xn, m=model, d=device: dl_predict_proba(m, Xn, d)
    expl    = shap.KernelExplainer(pred_fn, bg, link="identity")
    sv      = expl.shap_values(exp, nsamples=nsamples)
    arr     = np.stack([np.array(s) for s in sv], axis=0)
    if arr.shape[1] < arr.shape[2]:
        arr = arr.transpose(0, 2, 1)
    return np.mean(np.abs(arr).mean(axis=1), axis=0)

ref_mname = top3[0]
print(f"\n  Reference model: {ref_mname}")
print(f"  Running KernelSHAP across all {N_FOLDS} folds...\n")

fold_importances = []
for fold_idx, fd in enumerate(fold_data):
    print(f"  Fold {fold_idx+1} — retraining ...", end=" ", flush=True)
    m_f, _, _ = train_model(MODEL_REGISTRY[ref_mname], fd, DEVICE, ref_mname)
    m_f.eval()
    sc_sh   = StandardScaler()
    X_va_sh = sc_sh.fit_transform(fd["X_va"])
    print("SHAP ...", end=" ", flush=True)
    imp = run_shap_fold(m_f, X_va_sh, DEVICE)
    fold_importances.append(imp)
    top3_ch = [ch_labels_shap[i] for i in np.argsort(imp)[::-1][:3]]
    print(f"done.  Top-3: {top3_ch}")

fold_importances = np.array(fold_importances)
n_feats_sh       = fold_importances.shape[1]
pairs    = list(itertools.combinations(range(N_FOLDS), 2))
rho_vals = []
ranks    = np.argsort(np.argsort(-fold_importances, axis=1), axis=1)
for i, j in pairs:
    rho, _ = spearmanr(ranks[i], ranks[j])
    rho_vals.append(rho)
mean_rho = np.mean(rho_vals)
std_rho  = np.std(rho_vals)

print(f"\n  Spearman ρ (channel rank correlation):")
for (i, j), rho in zip(pairs, rho_vals):
    print(f"    Fold {i+1} vs Fold {j+1}: ρ = {rho:.4f}")
print(f"\n  Mean ρ = {mean_rho:.4f} ± {std_rho:.4f}")
print(f"  Stability: {'✓ Stable (ρ ≥ 0.70)' if mean_rho >= 0.70 else '✗ Unstable (ρ < 0.70)'}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"SHAP Attribution Stability — {ref_mname}\n"
             f"Mean Spearman ρ = {mean_rho:.4f}  (1.0 = perfectly stable across folds)",
             fontsize=12, fontweight="bold")
sorted_idx    = np.argsort(fold_importances.mean(axis=0))[::-1]
sorted_labels = [ch_labels_shap[i] for i in sorted_idx]
im0 = axes[0].imshow(fold_importances[:, sorted_idx], aspect="auto", cmap="YlOrRd")
axes[0].set_xticks(range(n_feats_sh))
axes[0].set_xticklabels(sorted_labels, rotation=90, fontsize=7)
axes[0].set_yticks(range(N_FOLDS))
axes[0].set_yticklabels([f"Fold {i+1}" for i in range(N_FOLDS)])
axes[0].set_title("Mean |SHAP| per Channel per Fold")
plt.colorbar(im0, ax=axes[0], label="|SHAP|")
rho_matrix = np.ones((N_FOLDS, N_FOLDS))
for (i, j), rho in zip(pairs, rho_vals):
    rho_matrix[i, j] = rho_matrix[j, i] = rho
im1 = axes[1].imshow(rho_matrix, vmin=0, vmax=1, cmap="RdYlGn")
axes[1].set_xticks(range(N_FOLDS)); axes[1].set_yticks(range(N_FOLDS))
axes[1].set_xticklabels([f"F{i+1}" for i in range(N_FOLDS)])
axes[1].set_yticklabels([f"F{i+1}" for i in range(N_FOLDS)])
axes[1].set_title("Spearman ρ Matrix\n(green = stable)")
plt.colorbar(im1, ax=axes[1], label="ρ")
for i in range(N_FOLDS):
    for j in range(N_FOLDS):
        axes[1].text(j, i, f"{rho_matrix[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
save_fig("FIX_04_shap_stability_across_folds")
print("[SHAP STABILITY] Complete.\n")

  SHAP ATTRIBUTION STABILITY ACROSS FOLDS

  Reference model: BiMamba-STAN
  Running KernelSHAP across all 5 folds...

  Fold 1 — retraining ... SHAP ... 

  0%|          | 0/80 [00:00<?, ?it/s]

done.  Top-3: ['X2', 'X3', 'X4']
  Fold 2 — retraining ... SHAP ... 

  0%|          | 0/80 [00:00<?, ?it/s]

done.  Top-3: ['X4', 'X3', 'X2']
  Fold 3 — retraining ... SHAP ... 

...

done.  Top-3: ['X2', 'X4', 'X3']
  Fold 4 — retraining ... SHAP ... 

  0%|          | 0/80 [00:00<?, ?it/s]

done.  Top-3: ['X2', 'X4', 'X3']
  Fold 5 — retraining ... SHAP ... 

  0%|          | 0/80 [00:00<?, ?it/s]

done.  Top-3: ['X4', 'X2', 'X3']

  Spearman ρ (channel rank correlation):
    Fold 1 vs Fold 2: ρ = 0.2000
    Fold 1 vs Fold 3: ρ = 0.8000
    Fold 1 vs Fold 4: ρ = 0.8000
    Fold 1 vs Fold 5: ρ = 0.4000
    Fold 2 vs Fold 3: ρ = 0.4000
    Fold 2 vs Fold 4: ρ = 0.4000
    Fold 2 vs Fold 5: ρ = 0.8000
    Fold 3 vs Fold 4: ρ = 1.0000
    Fold 3 vs Fold 5: ρ = 0.8000
    Fold 4 vs Fold 5: ρ = 0.8000

  Mean ρ = 0.6400 ± 0.2498
  Stability: ✗ Unstable (ρ < 0.70)
[SAVED] /content/outputs/FIX_04_shap_stability_across_folds.png
[SHAP STABILITY] Complete.



In [ ]:
# ── 23. DEMOGRAPHIC TRANSFER EVALUATION (Step 13 — original groups) ───────────
print("=" * 70)
print("  DEMOGRAPHIC TRANSFER EVALUATION — TOP 3 MODELS")
print("=" * 70)

groupA_mask = sub_id < 20
groupB_mask = sub_id >= 20
X_A_raw = X_raw[groupA_mask]; y_A = y_all[groupA_mask]
X_B_raw = X_raw[groupB_mask]; y_B = y_all[groupB_mask]
print(f"  Group A classes: {sorted(np.unique(y_A))}  ({len(y_A)} segments)")
print(f"  Group B classes: {sorted(np.unique(y_B))}  ({len(y_B)} segments)\n")

def safe_auroc(y_true, y_prob):
    present = np.unique(y_true)
    if len(present) < 2:
        return float("nan")
    aucs = []
    for cls in present:
        y_bin  = (y_true == cls).astype(int)
        score  = y_prob[:, cls] if y_prob.shape[1] > cls else y_prob[:, -1]
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            continue
        try:
            aucs.append(roc_auc_score(y_bin, score))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else float("nan")

transfer_rows = []
for mname, model in top3_model_objects.items():
    scaler_t = StandardScaler()
    X_A_s    = scaler_t.fit_transform(X_A_raw)
    X_B_s    = scaler_t.transform(X_B_raw)
    yp_A     = dl_predict_proba(model, X_A_s, DEVICE)
    yp_A_cls = yp_A.argmax(axis=1)
    f1_i  = f1_score(y_A, yp_A_cls, average="macro", zero_division=0)
    far_i = compute_far(y_A, yp_A_cls, len(y_A))
    auc_i = safe_auroc(y_A, yp_A)
    yp_B     = dl_predict_proba(model, X_B_s, DEVICE)
    yp_B_cls = yp_B.argmax(axis=1)
    f1_c  = f1_score(y_B, yp_B_cls, average="macro", zero_division=0)
    far_c = compute_far(y_B, yp_B_cls, len(y_B))
    auc_c = safe_auroc(y_B, yp_B)
    transfer_rows.append({
        "Model": mname,
        "F1 In-dist":  round(f1_i,  4), "F1 Cross":  round(f1_c,  4), "ΔF1":  round(f1_c  - f1_i,  4),
        "AUC In-dist": round(auc_i, 4), "AUC Cross": round(auc_c, 4), "ΔAUC": round(auc_c - auc_i, 4),
        "FAR In-dist": round(far_i, 3), "FAR Cross": round(far_c, 3), "ΔFAR": round(far_c - far_i, 3),
    })
    auc_i_s = f"{auc_i:.3f}" if not np.isnan(auc_i) else "n/a"
    auc_c_s = f"{auc_c:.3f}" if not np.isnan(auc_c) else "n/a"
    print(f"  {mname:15s}  F1: {f1_i:.3f}→{f1_c:.3f} (Δ{f1_c-f1_i:+.3f})  "
          f"AUC: {auc_i_s}→{auc_c_s}  FAR: {far_i:.2f}→{far_c:.2f}/hr")

df_transfer = pd.DataFrame(transfer_rows).set_index("Model")
print("\n  Note: BEED assigns one class per subject.")
print("    Group A (subjects 0–19) = classes 0 & 1 only.")
print("    Group B (subjects 20–39) = classes 2 & 3 only.")
print("[Transfer Summary]\n", df_transfer.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Demographic Transfer Evaluation — Top 3 Models\n"
             "(Group A = classes 0&1 → Group B = classes 2&3)", fontsize=12, fontweight="bold")
x = np.arange(len(df_transfer)); w = 0.35
model_labels = list(df_transfer.index)
for ax, (col_i, col_c, lbl) in zip(axes, [
    ("F1 In-dist", "F1 Cross",  "Macro F1"),
    ("AUC In-dist","AUC Cross", "AUROC (per-group OvR avg)"),
    ("FAR In-dist","FAR Cross", "FAR / hr"),
]):
    v_i = df_transfer[col_i].values.astype(float)
    v_c = df_transfer[col_c].values.astype(float)
    v_i_plot = np.nan_to_num(v_i, nan=0.0)
    v_c_plot = np.nan_to_num(v_c, nan=0.0)
    ax.bar(x-w/2, v_i_plot, w, label="In-dist (A→A)", color="#2E7D32", alpha=0.85, edgecolor="k")
    ax.bar(x+w/2, v_c_plot, w, label="Cross  (A→B)",  color="#C62828", alpha=0.85, edgecolor="k")
    ax.set_xticks(x); ax.set_xticklabels(model_labels, rotation=15, ha="right", fontsize=8)
    ax.set_title(lbl, fontsize=9); ax.legend(fontsize=8)
    if lbl == "FAR / hr":
        ax.axhline(FAR_THRESHOLD, color="red", ls="--", lw=1.5)
    for i2, (vi, vc) in enumerate(zip(v_i, v_c)):
        vi_s = f"{vi:.3f}" if not np.isnan(vi) else "n/a"
        vc_s = f"{vc:.3f}" if not np.isnan(vc) else "n/a"
        ax.text(i2-w/2, v_i_plot[i2]+0.005, vi_s, ha="center", fontsize=7)
        ax.text(i2+w/2, v_c_plot[i2]+0.005, vc_s, ha="center", fontsize=7)
plt.tight_layout()
save_fig("EXT_13_transfer_top3")
print("[TRANSFER] Done.\n")

  DEMOGRAPHIC TRANSFER EVALUATION — TOP 3 MODELS
  Group A classes: [np.int64(0), np.int64(1)]  (4000 segments)
  Group B classes: [np.int64(2), np.int64(3)]  (4000 segments)

  BiMamba-STAN     F1: 0.442→0.591 (Δ+0.149)  AUC: 0.978→0.962  FAR: 13.86→16.07/hr
  CNN              F1: 0.425→0.575 (Δ+0.150)  AUC: 0.994→0.937  FAR: 13.77→13.86/hr
  CNN-LSTM         F1: 0.412→0.553 (Δ+0.141)  AUC: 0.989→0.925  FAR: 17.46→26.51/hr

  Note: BEED assigns one class per subject.
    Group A (subjects 0–19) = classes 0 & 1 only.
    Group B (subjects 20–39) = classes 2 & 3 only.
[Transfer Summary]
               F1 In-dist  F1 Cross     ΔF1  AUC In-dist  AUC Cross    ΔAUC  FAR In-dist  FAR Cross   ΔFAR
Model                                                                                                    
BiMamba-STAN      0.4424    0.5911  0.1487       0.9785     0.9616 -0.0169        13.86     16.065  2.205
CNN               0.4247    0.5752  0.1505       0.9937     0.9369 -0.0569        13.77 

In [ ]:
# ── 24. CORRECTED TRANSFER EVALUATION ────────────────────────────────────────
print("=" * 70)
print("  TRANSFER EVALUATION — CORRECTED METHODOLOGY")
print("=" * 70)

groupA_idx_c, groupB_idx_c = [], []
for cls in range(4):
    cls_subjects = np.unique(sub_id[y_all == cls])
    half         = len(cls_subjects) // 2
    for idx in np.where(np.isin(sub_id, cls_subjects[:half]))[0]:
        groupA_idx_c.append(idx)
    for idx in np.where(np.isin(sub_id, cls_subjects[half:]))[0]:
        groupB_idx_c.append(idx)
groupA_idx_c = np.array(groupA_idx_c)
groupB_idx_c = np.array(groupB_idx_c)
X_Ac = X_raw[groupA_idx_c]; y_Ac = y_all[groupA_idx_c]
X_Bc = X_raw[groupB_idx_c]; y_Bc = y_all[groupB_idx_c]
print(f"  Group A: {len(groupA_idx_c)} segs  classes {sorted(np.unique(y_Ac))}")
print(f"  Group B: {len(groupB_idx_c)} segs  classes {sorted(np.unique(y_Bc))}\n")

def safe_multiclass_auroc(y_true, y_prob):
    try:
        return roc_auc_score(label_binarize(y_true, classes=[0,1,2,3]),
                             y_prob, multi_class="ovr", average="macro")
    except Exception:
        return float("nan")

transfer_rows_c = []
for mname, model in top3_model_objects.items():
    sc   = StandardScaler()
    Xa_s = sc.fit_transform(X_Ac)
    Xb_s = sc.transform(X_Bc)
    for split_name, Xs, ys in [("In-dist (A)", Xa_s, y_Ac), ("Cross (B)", Xb_s, y_Bc)]:
        prob = dl_predict_proba(model, Xs, DEVICE)
        pred = prob.argmax(1)
        transfer_rows_c.append({
            "Model": mname, "Split": split_name,
            "F1":    round(f1_score(ys, pred, average="macro", zero_division=0), 4),
            "AUROC": round(safe_multiclass_auroc(ys, prob), 4),
            "FAR":   round(compute_far(ys, pred, len(ys)), 3),
        })

df_tr = pd.DataFrame(transfer_rows_c)
print("[Corrected Transfer Results]")
print(df_tr.to_string(index=False))
print("\n[Transfer Gap Summary]")
for mname in top3:
    row_i = df_tr[(df_tr.Model==mname) & (df_tr.Split.str.contains("In-dist"))]
    row_c = df_tr[(df_tr.Model==mname) & (df_tr.Split.str.contains("Cross"))]
    if len(row_i) and len(row_c):
        df1 = row_c.iloc[0]["F1"]   - row_i.iloc[0]["F1"]
        da  = row_c.iloc[0]["AUROC"] - row_i.iloc[0]["AUROC"]
        print(f"  {mname:15s}  ΔF1={df1:+.4f}  ΔAUROC={da:+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Corrected Transfer Evaluation\n"
             "(Within-class subject split — all 4 classes present in both groups)",
             fontsize=12, fontweight="bold")
for ax, metric in zip(axes, ["F1", "AUROC", "FAR"]):
    x = np.arange(len(top3)); w = 0.35
    v_i = [df_tr[(df_tr.Model==m) & (df_tr.Split.str.contains("In"))][metric].values[0] for m in top3]
    v_c = [df_tr[(df_tr.Model==m) & (df_tr.Split.str.contains("Cross"))][metric].values[0] for m in top3]
    ax.bar(x-w/2, v_i, w, label="In-dist (A)", color="#2E7D32", alpha=0.85, edgecolor="k")
    ax.bar(x+w/2, v_c, w, label="Cross   (B)", color="#C62828", alpha=0.85, edgecolor="k")
    ax.set_xticks(x); ax.set_xticklabels(top3, rotation=15, ha="right", fontsize=8)
    ax.set_title(metric); ax.legend(fontsize=8)
    for i, (a, b) in enumerate(zip(v_i, v_c)):
        ax.text(i-w/2, a+0.005, f"{a:.3f}", ha="center", fontsize=7)
        ax.text(i+w/2, b+0.005, f"{b:.3f}", ha="center", fontsize=7)
plt.tight_layout()
save_fig("FIX_05_transfer_corrected")
print("[CORRECTED TRANSFER] Complete.\n")

  TRANSFER EVALUATION — CORRECTED METHODOLOGY
  Group A: 4000 segs  classes [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Group B: 4000 segs  classes [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

[Corrected Transfer Results]
       Model       Split     F1  AUROC    FAR
BiMamba-STAN In-dist (A) 0.8829 0.9754  9.585
BiMamba-STAN   Cross (B) 0.9321 0.9920  2.925
         CNN In-dist (A) 0.8987 0.9841  9.315
         CNN   Cross (B) 0.9397 0.9932  3.735
    CNN-LSTM In-dist (A) 0.8672 0.9717 12.060
    CNN-LSTM   Cross (B) 0.9227 0.9886  4.950

[Transfer Gap Summary]
  BiMamba-STAN     ΔF1=+0.0492  ΔAUROC=+0.0166
  CNN              ΔF1=+0.0410  ΔAUROC=+0.0091
  CNN-LSTM         ΔF1=+0.0555  ΔAUROC=+0.0169
[SAVED] /content/outputs/FIX_05_transfer_corrected.png
[CORRECTED TRANSFER] Complete.



In [ ]:
# ── 25. ABLATION STUDIES ──────────────────────────────────────────────────────
print("=" * 70)
print("  ABLATION STUDIES — TOP 3 MODELS")
print("=" * 70)

scaler_abl = StandardScaler()
scaler_abl.fit(fd5["X_tr"])
X_va_abl_s = scaler_abl.transform(fd5["X_va"])
y_va_abl   = fd5["y_va"]
top5_indices = [int(ch_labels_shap[i][1:]) - 1 for i in _rank_ref[:min(5, _n_feats_ref)]]
all_ablation_results = {}
rng_abl = np.random.default_rng(SEED)

for mname, model in top3_model_objects.items():
    print(f"\n  [{mname}]")
    abl_res = {}
    base_pred = dl_predict_proba(model, X_va_abl_s, DEVICE).argmax(axis=1)
    base_f1   = f1_score(y_va_abl, base_pred, average="macro", zero_division=0)
    abl_res["Base"] = base_f1
    print(f"    Base:           F1={base_f1:.3f}")
    X_shuf = X_va_abl_s.copy()
    for i in range(len(X_shuf)):
        X_shuf[i] = X_shuf[i, rng_abl.permutation(X_va_abl_s.shape[1])]
    shuf_f1 = f1_score(y_va_abl, dl_predict_proba(model, X_shuf, DEVICE).argmax(axis=1),
                       average="macro", zero_division=0)
    abl_res["Feat Shuffle"] = shuf_f1
    print(f"    Feat Shuffle:   F1={shuf_f1:.3f}  ΔF1={shuf_f1-base_f1:+.3f}")
    for ch_idx in top5_indices:
        X_drop = X_va_abl_s.copy(); X_drop[:, ch_idx] = 0.0
        df1 = f1_score(y_va_abl, dl_predict_proba(model, X_drop, DEVICE).argmax(axis=1),
                       average="macro", zero_division=0)
        label = f"Drop X{ch_idx+1}"
        abl_res[label] = df1
        print(f"    {label:14s}  F1={df1:.3f}  ΔF1={df1-base_f1:+.3f}")
    X_noisy  = X_va_abl_s + rng_abl.normal(0, 0.5, X_va_abl_s.shape)
    noise_f1 = f1_score(y_va_abl, dl_predict_proba(model, X_noisy, DEVICE).argmax(axis=1),
                        average="macro", zero_division=0)
    abl_res["Noise σ=0.5"] = noise_f1
    print(f"    Noise σ=0.5:    F1={noise_f1:.3f}  ΔF1={noise_f1-base_f1:+.3f}")
    all_ablation_results[mname] = abl_res

n_abl = len(all_ablation_results)
fig, axes = plt.subplots(1, n_abl, figsize=(7*n_abl, 6), sharey=False)
axes = [axes] if n_abl == 1 else list(axes)
fig.suptitle("Ablation Studies — Top 3 Models (ΔF1 vs Base)", fontsize=13, fontweight="bold")
for ax, (mname, abl_res) in zip(axes, all_ablation_results.items()):
    base   = abl_res["Base"]
    conds  = list(abl_res.keys())
    deltas = [abl_res[c] - base for c in conds]
    cols   = ["#2E7D32" if d >= 0 else "#C62828" for d in deltas]
    bars   = ax.bar(conds, deltas, color=cols, edgecolor="k", lw=0.7, alpha=0.85)
    ax.axhline(0, color="k", lw=1)
    ax.set_title(f"{mname}\n(Base F1={base:.3f})", fontsize=10)
    ax.set_ylabel("ΔMacro F1")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
    for bar, d in zip(bars, deltas):
        ax.text(bar.get_x()+bar.get_width()/2, d+(0.003 if d >= 0 else -0.008),
                f"{d:+.3f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
save_fig("EXT_14_ablation_top3")
print("\n[ABLATION] Done.\n")

  ABLATION STUDIES — TOP 3 MODELS

  [BiMamba-STAN]
    Base:           F1=0.788
    Feat Shuffle:   F1=0.543  ΔF1=-0.245
    Drop X2         F1=0.786  ΔF1=-0.002
    Drop X4         F1=0.767  ΔF1=-0.020
    Drop X3         F1=0.788  ΔF1=+0.000
    Drop X1         F1=0.758  ΔF1=-0.030
    Noise σ=0.5:    F1=0.440  ΔF1=-0.348

  [CNN]
    Base:           F1=0.776
    Feat Shuffle:   F1=0.381  ΔF1=-0.395
    Drop X2         F1=0.732  ΔF1=-0.044
    Drop X4         F1=0.746  ΔF1=-0.030
    Drop X3         F1=0.769  ΔF1=-0.007
    Drop X1         F1=0.753  ΔF1=-0.023
    Noise σ=0.5:    F1=0.443  ΔF1=-0.333

  [CNN-LSTM]
    Base:           F1=0.709
    Feat Shuffle:   F1=0.339  ΔF1=-0.370
    Drop X2         F1=0.698  ΔF1=-0.011
    Drop X4         F1=0.709  ΔF1=-0.000
    Drop X3         F1=0.707  ΔF1=-0.002
    Drop X1         F1=0.680  ΔF1=-0.029
    Noise σ=0.5:    F1=0.454  ΔF1=-0.255
[SAVED] /content/outputs/EXT_14_ablation_top3.png

[ABLATION] Done.



In [ ]:
# ── 26. FINAL MODEL SELECTION ─────────────────────────────────────────────────
print("=" * 70)
print("  FINAL MODEL SELECTION — TOP 3 → 1 BEST")
print("=" * 70)

df_top3_eval = df_eval.loc[top3].copy()
df_top3_eval["Transfer ΔF1"] = [
    df_transfer.loc[m, "ΔF1"] if m in df_transfer.index else float("nan") for m in top3
]
abl_robust = {}
for mname, abl_res in all_ablation_results.items():
    base   = abl_res["Base"]
    deltas = [v - base for k, v in abl_res.items() if k != "Base"]
    abl_robust[mname] = float(np.mean(deltas)) if deltas else 0.0
df_top3_eval["Ablation ΔF1"] = [abl_robust.get(m, float("nan")) for m in top3]

def norm_higher(s): r = s - s.min(); return r / (r.max() + 1e-9)
def norm_lower(s):  r = s.max() - s; return r / (r.max() + 1e-9)

df_top3_eval["F1_n"]    = norm_higher(df_top3_eval["Macro F1"])
df_top3_eval["AUC_n"]   = norm_higher(df_top3_eval["AUROC"])
df_top3_eval["FAR_n"]   = norm_lower (df_top3_eval["FAR/hr"])
df_top3_eval["Trans_n"] = norm_higher(df_top3_eval["Transfer ΔF1"])
df_top3_eval["Abl_n"]   = norm_higher(df_top3_eval["Ablation ΔF1"])
df_top3_eval["Composite"] = (df_top3_eval["F1_n"]    * 0.30 +
                              df_top3_eval["AUC_n"]   * 0.25 +
                              df_top3_eval["FAR_n"]   * 0.25 +
                              df_top3_eval["Trans_n"] * 0.10 +
                              df_top3_eval["Abl_n"]   * 0.10)
df_top3_eval = df_top3_eval.sort_values("Composite", ascending=False)
best_model   = df_top3_eval.index[0]
print("\n[Top 3 Comparison]")
print(df_top3_eval[["Macro F1","AUROC","FAR/hr","Transfer ΔF1","Ablation ΔF1","Composite"]].round(4).to_string())
print(f"\n  ★ FINAL BEST MODEL: {best_model}  (Composite={df_top3_eval.loc[best_model,'Composite']:.4f})")

radar_dims   = ["F1_n","AUC_n","FAR_n","Trans_n","Abl_n"]
radar_labels = ["F1","AUROC","FAR\n(inv)","Transfer\nRobust","Ablation\nRobust"]
N_r    = len(radar_dims)
angles = np.linspace(0, 2*np.pi, N_r, endpoint=False).tolist() + [0]
fig = plt.figure(figsize=(5*len(top3), 5))
fig.suptitle("Final Selection — Normalised Score Profiles", fontsize=13, fontweight="bold")
for i, mname in enumerate(df_top3_eval.index):
    ax   = fig.add_subplot(1, len(top3), i+1, projection="polar")
    vals = [df_top3_eval.loc[mname, d] for d in radar_dims] + [df_top3_eval.loc[mname, radar_dims[0]]]
    star = " ★" if mname == best_model else ""
    ax.plot(angles, vals, color=f"C{i}", lw=2)
    ax.fill(angles, vals, color=f"C{i}", alpha=0.2)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_labels, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{mname}{star}\nComposite={df_top3_eval.loc[mname,'Composite']:.3f}", fontsize=9, pad=14)
plt.tight_layout()
save_fig("EXT_15a_radar_top3")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Top 3 Final Metric Comparison", fontsize=13, fontweight="bold")
bar_cols_15 = ["gold" if m == best_model else "#546E7A" for m in df_top3_eval.index]
for ax, (col, lbl, tgt, hi) in zip(axes, [
    ("Macro F1", "Macro F1", 0.90, True),
    ("AUROC",    "AUROC",    0.95, True),
    ("FAR/hr",   "FAR / hr", FAR_THRESHOLD, False),
]):
    vals = df_top3_eval[col].values
    ax.bar(df_top3_eval.index, vals, color=bar_cols_15, edgecolor="k", lw=0.8, alpha=0.9)
    ax.axhline(tgt, color="green" if hi else "red", lw=2, ls="--", label=f"{'Target' if hi else 'Limit'} {tgt}")
    ax.set_title(lbl); ax.legend(fontsize=8)
    for j, v in enumerate(vals):
        ax.text(j, v+0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
save_fig("EXT_15b_bars_top3")
print("\n[FINAL SELECTION] Done.\n")

  FINAL MODEL SELECTION — TOP 3 → 1 BEST

[Top 3 Comparison]
              Macro F1   AUROC  FAR/hr  Transfer ΔF1  Ablation ΔF1  Composite
Model                                                                        
BiMamba-STAN    0.8039  0.9330  13.500        0.1487       -0.1075     0.9806
CNN             0.7765  0.9308  15.165        0.1505       -0.1388     0.7407
CNN-LSTM        0.7105  0.8916  20.678        0.1412       -0.1113     0.0878

  ★ FINAL BEST MODEL: BiMamba-STAN  (Composite=0.9806)
[SAVED] /content/outputs/EXT_15a_radar_top3.png
[SAVED] /content/outputs/EXT_15b_bars_top3.png

[FINAL SELECTION] Done.



In [ ]:
# ── 27. STATISTICAL TESTING ───────────────────────────────────────────────────
print("=" * 70)
print(f"  STATISTICAL TESTING — {best_model} vs ALL OTHERS")
print("=" * 70)

model_fold_f1 = {}
for mname in ["SVM", "RF", "XGBoost"]:
    f1s = df_classical[df_classical["model"] == mname]["macro_f1"].values
    if len(f1s): model_fold_f1[mname] = f1s
for mname in ["CNN", "CNN-LSTM", "Transformer", "BiMamba-STAN"]:
    f1s = df_dl[df_dl["model"] == mname]["macro_f1"].values
    if len(f1s): model_fold_f1[mname] = f1s

best_f1_arr = model_fold_f1.get(best_model)
if best_f1_arr is None:
    print(f"  No per-fold F1 found for {best_model}.")
else:
    print(f"\n  {best_model}  F1 per fold: {np.round(best_f1_arr, 4)}")
    print(f"  Mean={np.mean(best_f1_arr):.4f}  Std={np.std(best_f1_arr):.4f}\n")
    stat_rows = []
    for other, other_f1s in model_fold_f1.items():
        if other == best_model: continue
        n = min(len(best_f1_arr), len(other_f1s))
        a, b = best_f1_arr[:n], other_f1s[:n]
        try:    _, p_t = ttest_rel(a, b)
        except: p_t = float("nan")
        try:    _, p_w = wilcoxon(a, b)
        except: p_w = float("nan")
        sig = "✓ p<0.05" if p_t < 0.05 else "✗ n.s."
        w_note = "⚠ floor (n=5, p_min=0.0625)" if abs(p_w - 0.0625) < 0.001 else f"{p_w:.4f}"
        stat_rows.append({"vs": other, "Mean ΔF1": round(float(np.mean(a-b)), 4),
                           "t-test p": round(float(p_t), 4),
                           "Wilcoxon p": round(float(p_w), 4),
                           "Significant": sig, "Wilcoxon note": w_note})
        print(f"  vs {other:15s}  ΔF1={np.mean(a-b):+.4f}  t-p={p_t:.4f}  W-p={p_w:.4f}  {sig}")
    df_stat = pd.DataFrame(stat_rows)
    print(f"""
  Note on Wilcoxon with n={N_FOLDS} folds:
  The minimum achievable p-value is 1/2^(n-1) = 0.0625.
  Wilcoxon p=0.0625 does not establish significance at α=0.05.
  Interpret paired t-test results alongside fold-level effect sizes.
  """)

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.suptitle(f"Per-Fold Macro F1  |  Final: {best_model} ★", fontsize=12, fontweight="bold")
    for i, (mname, f1s) in enumerate(model_fold_f1.items()):
        is_best = (mname == best_model)
        ax.plot(range(1, len(f1s)+1), f1s, marker="o",
                lw=3 if is_best else 1.5, alpha=1.0 if is_best else 0.5,
                color=f"C{i}", zorder=5 if is_best else 3,
                label=f"{mname} (μ={np.mean(f1s):.3f}±{np.std(f1s):.3f})" + (" ★" if is_best else ""))
    ax.axhline(0.90, color="grey", ls=":", lw=1.5, label="0.90 target")
    ax.set_xlabel("Fold"); ax.set_ylabel("Macro F1"); ax.set_xticks(range(1, N_FOLDS+1))
    ax.legend(fontsize=8, bbox_to_anchor=(1.01,1), loc="upper left"); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_fig("EXT_16a_fold_f1_final")

    if not df_stat.empty:
        fig, ax = plt.subplots(figsize=(10, max(3, len(df_stat)*0.65+1.5)))
        fig.suptitle(f"Statistical Significance  |  {best_model} vs All", fontsize=12, fontweight="bold")
        cols_s = ["#2E7D32" if "✓" in r else "#90A4AE" for r in df_stat["Significant"]]
        bars   = ax.barh(df_stat["vs"], df_stat["Mean ΔF1"], color=cols_s, edgecolor="k", lw=0.7)
        ax.axvline(0, color="k", lw=0.8)
        ax.set_xlabel(f"Mean ΔF1  ({best_model} minus competitor)")
        ax.set_title("Green = significant (p<0.05 paired t-test)")
        for bar, p, sig in zip(bars, df_stat["t-test p"], df_stat["Significant"]):
            ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                    f"p={p:.3f} {sig}", va="center", fontsize=8)
        ax.invert_yaxis()
        plt.tight_layout()
        save_fig("EXT_16b_significance")

print(f"\n{'='*70}")
print(f"  FINAL BEST MODEL : {best_model}")
print(f"  Macro F1  = {df_top3_eval.loc[best_model,'Macro F1']:.4f}")
print(f"  AUROC     = {df_top3_eval.loc[best_model,'AUROC']:.4f}")
print(f"  FAR/hr    = {df_top3_eval.loc[best_model,'FAR/hr']:.2f}")
print(f"  Composite = {df_top3_eval.loc[best_model,'Composite']:.4f}")
print(f"{'='*70}\n")

  STATISTICAL TESTING — BiMamba-STAN vs ALL OTHERS

  BiMamba-STAN  F1 per fold: [0.7915 0.8454 0.7765 0.8084 0.7912]
  Mean=0.8026  Std=0.0237

  vs SVM              ΔF1=+0.2195  t-p=0.0000  W-p=0.0625  ✓ p<0.05
  vs RF               ΔF1=+0.2055  t-p=0.0002  W-p=0.0625  ✓ p<0.05
  vs XGBoost          ΔF1=+0.1955  t-p=0.0002  W-p=0.0625  ✓ p<0.05
  vs CNN              ΔF1=+0.0265  t-p=0.1925  W-p=0.1250  ✗ n.s.
  vs CNN-LSTM         ΔF1=+0.0962  t-p=0.0026  W-p=0.0625  ✓ p<0.05
  vs Transformer      ΔF1=+0.2787  t-p=0.0002  W-p=0.0625  ✓ p<0.05

  Note on Wilcoxon with n=5 folds:
  The minimum achievable p-value is 1/2^(n-1) = 0.0625.
  Wilcoxon p=0.0625 does not establish significance at α=0.05.
  Interpret paired t-test results alongside fold-level effect sizes.
  
[SAVED] /content/outputs/EXT_16a_fold_f1_final.png
[SAVED] /content/outputs/EXT_16b_significance.png

  FINAL BEST MODEL : BiMamba-STAN
  Macro F1  = 0.8039
  AUROC     = 0.9330
  FAR/hr    = 13.50
  Composite = 0.9806



In [ ]:
# ── 28. INFERENCE TIME PER WINDOW ────────────────────────────────────────────
print("=" * 70)
print("  INFERENCE TIME PER WINDOW (Objective 5)")
print("=" * 70)

OBJECTIVE_LATENCY_S = 10.0

def measure_inference_time(model, X_np, device, n_repeats=50):
    model.eval()
    x_single = torch.tensor(X_np[[0]], dtype=torch.float32).unsqueeze(1).to(device)
    with torch.no_grad():
        for _ in range(10):
            _ = model(x_single)
    times = []
    with torch.no_grad():
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            _ = model(x_single)
            times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times), np.min(times), np.max(times)

fd5        = fold_data[4]
scaler_inf = StandardScaler()
X_inf      = scaler_inf.fit_transform(fd5["X_tr"])
inference_rows = []
for mname, model in top3_model_objects.items():
    mean_s, std_s, min_s, max_s = measure_inference_time(model, X_inf, DEVICE)
    passed = "✓" if mean_s < OBJECTIVE_LATENCY_S else "✗"
    inference_rows.append({
        "Model": mname, "Mean (ms)": round(mean_s * 1000, 3), "Std  (ms)": round(std_s  * 1000, 3),
        "Min  (ms)": round(min_s  * 1000, 3), "Max  (ms)": round(max_s  * 1000, 3), "< 10 s ✓": passed,
    })
    print(f"  {mname:15s}  {mean_s*1000:.3f} ms ± {std_s*1000:.3f} ms   {passed}")

df_inference = pd.DataFrame(inference_rows).set_index("Model")
print("\n[Inference Latency Table]")
print(df_inference.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle(f"Per-Window Inference Latency — Top 3 Models\nObjective: < {OBJECTIVE_LATENCY_S} s per window",
             fontsize=12, fontweight="bold")
models_inf = list(df_inference.index)
means_ms   = df_inference["Mean (ms)"].values
stds_ms    = df_inference["Std  (ms)"].values
bar_colors = ["#2E7D32" if r == "✓" else "#C62828" for r in df_inference["< 10 s ✓"]]
bars = ax.bar(models_inf, means_ms, yerr=stds_ms, capsize=6,
              color=bar_colors, edgecolor="k", lw=0.8, alpha=0.88)
ax.axhline(OBJECTIVE_LATENCY_S * 1000, color="red", lw=2, ls="--",
           label=f"Objective limit ({OBJECTIVE_LATENCY_S*1000:.0f} ms)")
green_p = mpatches.Patch(color="#2E7D32", label="Passes objective")
red_p   = mpatches.Patch(color="#C62828", label="Fails objective")
lim_l   = plt.Line2D([0],[0], color="red", ls="--", lw=2, label="10 s limit")
ax.legend(handles=[green_p, red_p, lim_l], fontsize=8)
ax.set_ylabel("Mean Latency (ms)")
for bar, mean, std in zip(bars, means_ms, stds_ms):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.02,
            f"{mean:.3f} ms", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
save_fig("FIX_01_inference_latency")
print("[INFERENCE TIME] Complete.\n")

  INFERENCE TIME PER WINDOW (Objective 5)
  BiMamba-STAN     3.549 ms ± 1.038 ms   ✓
  CNN              0.457 ms ± 0.041 ms   ✓
  CNN-LSTM         0.837 ms ± 0.101 ms   ✓

[Inference Latency Table]
              Mean (ms)  Std  (ms)  Min  (ms)  Max  (ms) < 10 s ✓
Model                                                            
BiMamba-STAN      3.549      1.038      2.202      6.246        ✓
CNN               0.457      0.041      0.398      0.627        ✓
CNN-LSTM          0.837      0.101      0.680      1.349        ✓
[SAVED] /content/outputs/FIX_01_inference_latency.png
[INFERENCE TIME] Complete.



In [ ]:
# ── 29. FAR POST-PROCESSING ───────────────────────────────────────────────────
print("=" * 70)
print("  FAR POST-PROCESSING (Objective 1)")
print("=" * 70)

def temporal_smoothing(y_pred, window=3, min_votes=2):
    n      = len(y_pred)
    result = y_pred.copy()
    for i in range(n):
        lo  = max(0, i - window // 2)
        hi  = min(n, lo + window)
        seg = y_pred[lo:hi]
        if np.sum(seg == 2) >= min_votes:
            result[i] = 2
        else:
            non_ictal = seg[seg != 2]
            result[i] = (np.bincount(non_ictal).argmax() if len(non_ictal) > 0 else seg[0])
    return result

def confidence_threshold(y_pred, y_prob, ictal_thr=0.6):
    result = y_pred.copy()
    for i in np.where(y_pred == 2)[0]:
        if y_prob[i, 2] < ictal_thr:
            remaining    = y_prob[i].copy()
            remaining[2] = -np.inf
            result[i]    = np.argmax(remaining)
    return result

best_data = all_model_preds[best_model]
yt  = best_data["y_true"]
yp  = best_data["y_pred"]
ypr = np.array(best_data["y_prob"])
base_far = compute_far(yt, yp, len(yt))
base_f1  = f1_score(yt, yp, average="macro", zero_division=0)
print(f"  Baseline (no post-proc):  F1={base_f1:.4f}  FAR={base_far:.3f}/hr\n")

windows    = [1, 3, 5, 7]
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
pp_rows = []
for w in windows:
    for thr in thresholds:
        yp_sm = temporal_smoothing(yp.copy(), window=w, min_votes=max(1, w//2 + 1))
        yp_pp = confidence_threshold(yp_sm, ypr, ictal_thr=thr)
        f1_pp  = f1_score(yt, yp_pp, average="macro", zero_division=0)
        far_pp = compute_far(yt, yp_pp, len(yt))
        pp_rows.append({"Window": w, "Threshold": thr, "F1": round(f1_pp, 4), "FAR/hr": round(far_pp, 3)})

df_pp     = pd.DataFrame(pp_rows)
best_pp   = df_pp.loc[df_pp["FAR/hr"].idxmin()]
pct_reduc = (1 - best_pp["FAR/hr"] / base_far) * 100
print(f"  Best configuration: Window = {best_pp['Window']}, Threshold = {best_pp['Threshold']}")
print(f"  F1 = {best_pp['F1']:.4f}  FAR = {best_pp['FAR/hr']:.3f}/hr")
print(f"  FAR reduction vs baseline: {pct_reduc:.1f}%")
print(f"\n  FAR/hr grid (rows=window, cols=threshold):")
pivot = df_pp.pivot(index="Window", columns="Threshold", values="FAR/hr")
print(pivot.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"FAR Post-Processing Grid — {best_model}\n"
             f"Baseline FAR={base_far:.2f}/hr → Best={best_pp['FAR/hr']:.2f}/hr "
             f"({pct_reduc:.0f}% reduction)   |   Objective: ≤0.5/hr",
             fontsize=11, fontweight="bold")
pv_far = df_pp.pivot(index="Window", columns="Threshold", values="FAR/hr")
pv_f1  = df_pp.pivot(index="Window", columns="Threshold", values="F1")
for ax, pv, title, cmap, label in zip(
        axes, [pv_far, pv_f1],
        ["FAR/hr (lower = better)", "Macro F1 (higher = better)"],
        ["RdYlGn_r", "RdYlGn"], ["FAR/hr", "F1"]):
    im = ax.imshow(pv.values, aspect="auto", cmap=cmap, vmin=pv.values.min(), vmax=pv.values.max())
    ax.set_xticks(range(len(thresholds)))
    ax.set_xticklabels([str(t) for t in thresholds], fontsize=8)
    ax.set_yticks(range(len(windows)))
    ax.set_yticklabels([str(w) for w in windows])
    ax.set_xlabel("Confidence Threshold"); ax.set_ylabel("Smoothing Window"); ax.set_title(title)
    plt.colorbar(im, ax=ax, label=label)
    for i in range(len(windows)):
        for j in range(len(thresholds)):
            ax.text(j, i, f"{pv.values[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.tight_layout()
save_fig("FIX_06_far_postprocessing")
print("[FAR POST-PROCESSING] Complete.\n")

  FAR POST-PROCESSING (Objective 1)
  Baseline (no post-proc):  F1=0.8039  FAR=13.500/hr

  Best configuration: Window = 7.0, Threshold = 0.8
  F1 = 0.8003  FAR = 6.975/hr
  FAR reduction vs baseline: 48.3%

  FAR/hr grid (rows=window, cols=threshold):
Threshold    0.50    0.55    0.60    0.65   0.70   0.75   0.80
Window                                                        
1          12.892  11.970  11.160  10.418  9.788  8.955  7.762
3          12.532  11.722  10.912  10.215  9.630  8.798  7.628
5          11.362  10.642  10.080   9.472  9.022  8.370  7.335
7          10.642   9.945   9.495   8.955  8.572  7.965  6.975
[SAVED] /content/outputs/FIX_06_far_postprocessing.png
[FAR POST-PROCESSING] Complete.



In [ ]:
# ── 30. C0/C3 MISCLASSIFICATION REDUCTION ────────────────────────────────────
print("=" * 70)
print("  C0 ↔ C3 MISCLASSIFICATION % REDUCTION (Objective 3)")
print("=" * 70)

OBJECTIVE_REDUCTION = 25.0
fft_models    = ["SVM", "RF", "XGBoost"]
dl_models_c03 = [m for m in top3 if m in all_model_preds]
c03_rows = []
for mname, data in all_model_preds.items():
    yt = np.array(data["y_true"]); yp = np.array(data["y_pred"])
    mask      = (yt == 0) | (yt == 3)
    c0_as_c3  = int(np.sum((yp[mask] == 3) & (yt[mask] == 0)))
    c3_as_c0  = int(np.sum((yp[mask] == 0) & (yt[mask] == 3)))
    total_03  = int(np.sum(mask))
    conf_rate = (c0_as_c3 + c3_as_c0) / total_03 if total_03 > 0 else np.nan
    c03_rows.append({
        "Model": mname,
        "Group": "FFT Classical" if mname in fft_models else "DL Temporal",
        "C0→C3": c0_as_c3, "C3→C0": c3_as_c0,
        "Total C0+C3": total_03, "Conf Rate": round(conf_rate, 4),
    })

df_c03 = pd.DataFrame(c03_rows).set_index("Model")
fft_baseline = df_c03.loc[fft_models, "Conf Rate"].mean()
fft_best     = df_c03.loc[fft_models, "Conf Rate"].min()
print(f"\n  FFT baseline (mean classical): {fft_baseline:.4f}")
print(f"  FFT baseline (best classical): {fft_best:.4f}\n")
for mname in dl_models_c03:
    dl_rate     = df_c03.loc[mname, "Conf Rate"]
    pct_vs_mean = (fft_baseline - dl_rate) / fft_baseline * 100
    pct_vs_best = (fft_best     - dl_rate) / fft_best     * 100
    passed_mean = "✓" if pct_vs_mean >= OBJECTIVE_REDUCTION else "✗"
    df_c03.loc[mname, "% Red vs Mean FFT"] = round(pct_vs_mean, 2)
    df_c03.loc[mname, "% Red vs Best FFT"] = round(pct_vs_best, 2)
    df_c03.loc[mname, f"≥{OBJECTIVE_REDUCTION}%"] = passed_mean
    print(f"  {mname:15s}  Conf Rate={dl_rate:.4f}  "
          f"Δ vs mean={pct_vs_mean:+.1f}% {passed_mean}  Δ vs best={pct_vs_best:+.1f}%")

print("\n[C0/C3 Table]")
print(df_c03[["Group","C0→C3","C3→C0","Conf Rate","% Red vs Mean FFT","% Red vs Best FFT"]].round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"C0 ↔ C3 Misclassification Reduction\n"
             f"Objective: ≥{OBJECTIVE_REDUCTION}% reduction vs FFT baseline",
             fontsize=12, fontweight="bold")
all_m  = list(df_c03.index)
rates  = df_c03["Conf Rate"].values
cols   = ["#1565C0" if "FFT" in str(g) else "#6A1B9A" for g in df_c03["Group"].fillna("DL")]
axes[0].bar(all_m, rates, color=cols, edgecolor="k", lw=0.7, alpha=0.85)
axes[0].axhline(fft_baseline, color="red",    lw=2, ls="--", label=f"FFT mean ({fft_baseline:.4f})")
axes[0].axhline(fft_best,     color="orange", lw=2, ls=":",  label=f"FFT best ({fft_best:.4f})")
axes[0].set_ylabel("C0↔C3 Confusion Rate")
axes[0].set_title("Confusion Rate by Model")
axes[0].legend(fontsize=8)
plt.setp(axes[0].get_xticklabels(), rotation=25, ha="right")
dl_names = [m for m in dl_models_c03
            if "% Red vs Mean FFT" in df_c03.columns
            and not np.isnan(df_c03.loc[m, "% Red vs Mean FFT"])]
dl_pcts  = df_c03.loc[dl_names, "% Red vs Mean FFT"].values.astype(float)
bar_c2   = ["#2E7D32" if p >= OBJECTIVE_REDUCTION else "#C62828" for p in dl_pcts]
axes[1].bar(dl_names, dl_pcts, color=bar_c2, edgecolor="k", lw=0.7, alpha=0.85)
axes[1].axhline(OBJECTIVE_REDUCTION, color="green", lw=2, ls="--", label=f"{OBJECTIVE_REDUCTION}% objective")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("% Reduction vs Mean FFT")
axes[1].set_title("% Reduction in C0↔C3 Confusion")
axes[1].legend(fontsize=9)
for i, (nm, pct) in enumerate(zip(dl_names, dl_pcts)):
    axes[1].text(i, pct + 0.5, f"{pct:+.1f}%", ha="center", fontsize=10)
plt.tight_layout()
save_fig("FIX_02_c03_confusion_reduction")
print("[C0/C3 REDUCTION] Complete.\n")

  C0 ↔ C3 MISCLASSIFICATION % REDUCTION (Objective 3)

  FFT baseline (mean classical): 0.0037
  FFT baseline (best classical): 0.0018

  BiMamba-STAN     Conf Rate=0.0020  Δ vs mean=+45.9% ✓  Δ vs best=-11.1%
  CNN              Conf Rate=0.0020  Δ vs mean=+45.9% ✓  Δ vs best=-11.1%
  CNN-LSTM         Conf Rate=0.0010  Δ vs mean=+73.0% ✓  Δ vs best=+44.4%

[C0/C3 Table]
                      Group  C0→C3  C3→C0  Conf Rate  % Red vs Mean FFT  % Red vs Best FFT
Model                                                                                     
SVM           FFT Classical     16      4     0.0050                NaN                NaN
RF            FFT Classical      2      5     0.0018                NaN                NaN
XGBoost       FFT Classical      5     12     0.0043                NaN                NaN
CNN             DL Temporal      8      0     0.0020              45.95             -11.11
CNN-LSTM        DL Temporal      4      0     0.0010              72.97          

In [ ]:
# ── 31. RESULTS PACKAGING ────────────────────────────────────────────────────
print("=" * 70)
print("  RESULTS PACKAGING")
print("=" * 70)

PKG_DIR = OUT_DIR / "results_package"
PKG_DIR.mkdir(parents=True, exist_ok=True)

df_eval.to_csv(PKG_DIR / "full_evaluation_metrics.csv")
df_far.to_csv( PKG_DIR / "false_alarm_analysis.csv")
df_top3_eval.to_csv(PKG_DIR / "top3_final_selection.csv")
if "df_stat" in dir() and not df_stat.empty:
    df_stat.to_csv(PKG_DIR / "statistical_tests.csv", index=False)
df_transfer.to_csv(PKG_DIR / "transfer_evaluation.csv")
print("[PKG] Metrics tables saved.")

if best_model in top3_model_objects:
    torch.save(top3_model_objects[best_model].state_dict(),
               PKG_DIR / f"best_model_{best_model.replace('-','_')}.pt")
    print(f"[PKG] Best model ({best_model}) saved as .pt")

log = {
    "dataset":          "BEED_Data.csv",
    "n_segments":       int(len(y_all)),
    "n_subjects":       40,
    "n_channels":       int(N_CHANNELS),
    "n_folds":          int(N_FOLDS),
    "seed":             int(SEED),
    "best_model":       best_model,
    "best_macro_f1":    float(df_top3_eval.loc[best_model, "Macro F1"]),
    "best_auroc":       float(df_top3_eval.loc[best_model, "AUROC"]),
    "best_far_hr":      float(df_top3_eval.loc[best_model, "FAR/hr"]),
    "clinical_f1_met":  bool(df_top3_eval.loc[best_model, "Macro F1"] >= 0.90),
    "clinical_far_met": bool(df_top3_eval.loc[best_model, "FAR/hr"]   <= FAR_THRESHOLD),
    "top3_models":      top3,
    "top5_channels":    top5_ch,
    "transfer_delta_f1": float(df_transfer.loc[best_model, "ΔF1"]) if best_model in df_transfer.index else None,
}
with open(PKG_DIR / "experiment_log.json", "w") as fh:
    json.dump(log, fh, indent=2)
print("[PKG] Experiment log saved.")

all_figs = sorted(OUT_DIR.glob("*.png"))
print(f"\n[PKG] {len(all_figs)} figures in {OUT_DIR}:")
for f in all_figs:
    print(f"  {f.name}")

print(f"\n[PKG] Package contents in {PKG_DIR}:")
for f in sorted(PKG_DIR.iterdir()):
    print(f"  {f.name}")

ZIP_PATH = Path("/content/all_figures.zip") if Path("/content").exists() else OUT_DIR / "all_figures.zip"
png_files = sorted(OUT_DIR.rglob("*.png"))
if png_files:
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
        for png in png_files:
            arcname = png.relative_to(OUT_DIR)
            zf.write(png, arcname=arcname)
    size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)
    print(f"\n  {len(png_files)} figures → {ZIP_PATH.name}  ({size_mb:.2f} MB)")

try:
    from google.colab import files as _cf
    _cf.download(str(ZIP_PATH))
    print("  Download started.")
except Exception:
    print(f"  Zip saved at: {ZIP_PATH}")

print(f"\n{'='*70}")
print(f"  PIPELINE COMPLETE")
print(f"  Best model : {best_model}")
print(f"  Macro F1   = {df_top3_eval.loc[best_model,'Macro F1']:.4f}")
print(f"  AUROC      = {df_top3_eval.loc[best_model,'AUROC']:.4f}")
print(f"  FAR/hr     = {df_top3_eval.loc[best_model,'FAR/hr']:.2f}")
print(f"{'='*70}")

  RESULTS PACKAGING
[PKG] Metrics tables saved.
[PKG] Best model (BiMamba-STAN) saved as .pt
[PKG] Experiment log saved.

[PKG] 37 figures in /content/outputs:
  ANNOT_01_seizure_onset_single_channel.png
  ANNOT_02_all_classes_with_onset.png
  EDA_01_class_distribution.png
  EDA_02_channel_amplitude_per_class.png
  EDA_03_amplitude_violin.png
  EDA_04_correlation_heatmaps.png
  EDA_05_subject_level_summary.png
  EDA_06_channel_boxplots.png
  EDA_07_kurtosis_skewness.png
  EXT_09_false_alarm_analysis.png
  EXT_10a_confusion_matrices_all_models.png
  EXT_10b_roc_curves_seizure.png
  EXT_11_model_selection_radar.png
  EXT_12a_shap_global_importance_top3.png
  EXT_12b_shap_heatmap_top3.png
  EXT_12c_dot_BiMamba_STAN.png
  EXT_12c_dot_CNN.png
  EXT_12c_dot_CNN_LSTM.png
  EXT_13_transfer_top3.png
  EXT_14_ablation_top3.png
  EXT_15a_radar_top3.png
  EXT_15b_bars_top3.png
  EXT_16a_fold_f1_final.png
  EXT_16b_significance.png
  FIX_01_inference_latency.png
  FIX_02_c03_confusion_reduction.png

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Download started.

  PIPELINE COMPLETE
  Best model : BiMamba-STAN
  Macro F1   = 0.8039
  AUROC      = 0.9330
  FAR/hr     = 13.50
